# Korea RIM **Batch** Valuation v1.1
### Sales-driven DuPont ROE → Ohlson (1995) Residual Income Valuation

KRX universe 전체(또는 지정 범위)를 대상으로 RIM 평가를 **대량 배치 실행**하고
결과를 **DB(`korea_rim_valuation`)** 에 저장하는 노트북.

- **모든 입력 변수는 Cell 2 한 곳**에 집중 — Cell 2 만 수정하고 나머지는 위→아래 순서대로 실행
- 단일 종목 진단/Excel 출력 셀은 제거 → 배치 전용
  (개별 평가는 `korea_rim_individual_valuation_v1` 사용)

---

### ★ v5 → v1.1 핵심 수정 : 적정주가가 2개 나오던 원인 제거

구 노트북은 **클래스 셀 말미의 `TEST_TICKER` 실행이 v5 패치 셀보다 먼저** 돌아,
같은 종목이 v4 기하 decay(ω)로 한 번, 패치 후 진단 셀에서 v5 선형 fade 로 또 한 번 평가됐습니다.
(A204620: v4 TP **6,845원** / v5 TP **7,540원**)

**v1.1 조치** — v5 선형 fade 를 **클래스 본문에 확정**하고 패치 셀 삭제,
클래스 셀 내부의 테스트 실행·`plot()` 재평가 제거.
이제 평가 경로는 배치 셀 하나뿐이며 실행 순서와 무관하게 결과가 동일합니다.

```
spread(t) = spread₀ × (1 − t/N)      → CAP 종료 시 정확히 0
b(t)      = b₀ + (b_term − b₀)×t/N   (b_term = g_term / Re)
→ 마지막 해 RI = 0  ⇒  Terminal Value 구조적 0
```

## Cell 1 · 경로 자동 감지 (노트북/데스크탑 공용)

In [ ]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────
#  프로젝트 루트(= DATA 폴더의 부모)를 sys.path 에 등록.
#  → 노트북/데스크탑 어느 PC 에서 실행해도 from DATA.* import 가 동일하게 동작
#  ▸ 노트북   : C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast\DATA
#  ▸ 데스크탑 : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA
# ─────────────────────────────────────────────────────────────
_CANDIDATE_ROOTS = [
    r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast",
    r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy",
]

def _setup_path() -> str:
    try:
        start = Path(__file__).resolve().parent
    except NameError:
        start = Path.cwd()
    for p in [start] + list(start.parents):
        if (p / "DATA").is_dir():
            root = str(p)
            if root not in sys.path:
                sys.path.insert(0, root)
            print(f"[PATH] root 자동 감지 : {root}")
            return root
    for cand in _CANDIDATE_ROOTS:
        if os.path.isdir(cand) and os.path.isdir(os.path.join(cand, "DATA")):
            if cand not in sys.path:
                sys.path.insert(0, cand)
            print(f"[PATH] root 후보 경로 : {cand}")
            return cand
    raise EnvironmentError(
        "DATA 폴더를 찾을 수 없습니다. _CANDIDATE_ROOTS 를 환경에 맞게 수정하세요."
    )

_ROOT = _setup_path()
print(f"[확인] 프로젝트 루트 : {_ROOT}")
print(f"[확인] DATA 경로    : {os.path.join(_ROOT, 'DATA')}")


## Cell 2 · ★ 입력 변수 (여기만 수정하세요)

- `TICKER_START / TICKER_END / SKIP_DONE` : 배치 범위·이어하기
- `RUN_TICKERS_OVERRIDE` : 특정 종목만 배치 실행할 때
- `TOP_N / RANK_DATES_INPUT` : 배치 후 랭킹 조회 설정
- 이하 모델 파라미터 · Moat 6-Tier 등급표 · v1.1 가드 상수

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  ★★★ 입력 변수 셀 — 이 노트북의 모든 사용자 입력은 여기 한 곳 ★★★
#  (아래 셀들은 수정할 필요가 없습니다)
# ═══════════════════════════════════════════════════════════════

# ─────────────────────────────────────────────────────────────
#  ① 배치 실행 범위
# ─────────────────────────────────────────────────────────────
TICKER_START = 0          # universe 리스트에서 시작 인덱스 (0 = 처음부터)
TICKER_END   = 9999       # universe 리스트에서 끝 인덱스 (9999 = 끝까지)
SKIP_DONE    = False      # True → 체크포인트(done_tickers.txt) 기록 종목 건너뜀

RUN_TICKERS_OVERRIDE = [] # 특정 종목만 배치 실행할 때 사용. 비워두면([]) 전체 슬라이스
                          # 예: ["A204620", "A005930"]

VERBOSE_BATCH = False     # True → 종목별 상세 로그 (수천 종목 배치 시 False 권장)

# ─────────────────────────────────────────────────────────────
#  ② 랭킹 조회 (배치 완료 후 마지막 셀에서 사용)
# ─────────────────────────────────────────────────────────────
TOP_N            = 600    # upside 상위 추출 종목 수
RANK_DATES_INPUT = []     # 랭킹에 사용할 측정일 리스트.
                          # []  → 가장 최근 측정일 1일 자동 사용
                          # 예: ["2026-07-31", "2026-07-01"]
RANK_CSV_DIR     = (Path(r"C:/reports") if os.name == "nt"
                    else Path.home() / "reports")   # 랭킹 CSV 저장 폴더
SAVE_RANK_CSV    = True   # True → 랭킹 결과 CSV 저장

# ─────────────────────────────────────────────────────────────
#  ③ DB 테이블 (변경 시에만 수정)
# ─────────────────────────────────────────────────────────────
TABLE_FS        = "korea_fs_data_from_DG"             # 재무제표 원본 (DataGuide)
TABLE_FORECAST  = "korea_revenue_forecast_result"     # 매출 예측 결과
TABLE_MARKETCAP = "ks_listed_company_daily_marketcap" # 일별 시가총액
TABLE_PRICE     = "KSE_Price"                         # 일별 주가
TABLE_RESULT    = "korea_rim_valuation"               # ★ 결과 저장 테이블
TABLE_QUALITY   = "korea_valuation_quality_log"       # 데이터 품질 로그

# ─────────────────────────────────────────────────────────────
#  ④ 모델 파라미터
# ─────────────────────────────────────────────────────────────
FORECAST_HORIZON     = 8              # 매출 예측 사용 분기 수 (8분기 = 2년)
MIN_HISTORY          = 12             # DuPont OLS 최소 분기 수
MIN_REVENUE_QUARTERS = 24             # universe/평가 최소 조건 (24 = 6년)
OLS_MIN_R2           = 0.3            # 비율 회귀 채택 최소 R^2
OLS_MIN_SAMPLES      = 12             # 비율 회귀 채택 최소 표본 수
WINSORIZE_LIMITS     = (0.05, 0.95)   # 비율 계수 추정 시 winsorize 범위

GDP_GROWTH           = 0.04    # 한국 장기 GDP 성장률 (= Phase2 말기 BV 성장률 목표)
TV_RE_GAP            = 0.010   # (호환 유지) 구 TV 공식용. v5 는 RI_last=0 → TV=0 이라 미사용
RETENTION_FLOOR      = -0.50   # 유보율 하한 (buyback > NI 예외 케이스 허용)
BV_RETENTION_CAP     = 0.75    # Phase2 시작 유보율 b0 상한

# ─────────────────────────────────────────────────────────────
#  ⑤ ★ Moat 6-Tier 등급표 (v1.1: 클래스 내부 → 여기로 이관)
#     (등급, omega, Phase2 년수, EVA spread 임계, ROE-Re spread 임계, n_pos 최소)
#     · omega : moat 등급 메타데이터. ★ v5 선형 fade 에서는 감쇠 계산에 미사용
#               (Phase 2 는 년수 N 만으로 spread 를 선형 소멸시킴)
#     · 1차 분류는 (EVA spread > 임계) OR (ROE-Re spread > 임계)
#     · n_pos(20분기 중 양(+) EVA 분기 수)가 최소치 미달이면 한 등급 강등
# ─────────────────────────────────────────────────────────────
MOAT_TIERS = [
    ("Exceptional moat", 0.990, 30, 0.25, 1.50, 18),
    ("Wide moat",        0.980, 28, 0.15, 0.80, 16),
    ("Wide-Narrow moat", 0.970, 25, 0.10, 0.40, 13),
    ("Narrow moat",      0.950, 20, 0.06, 0.20, 10),
    ("Some moat",        0.940, 15, 0.03, 0.05,  7),
    ("No moat",          0.880, 10, float("-inf"), float("-inf"), 0),
]
MOAT_UNKNOWN_OMEGA  = 0.760   # EVA/ROE 스프레드 둘 다 산출 실패 시 fallback
MOAT_UNKNOWN_PHASE2 = 8

# ── Phase 1 기간: moat 등급별 plateau 연장 ────────────────────
PHASE1_BASE_YR  = 2   # 매출 forecast 실제 예측 2년 (고정)
PHASE1_EXTRA_YR = {
    "Exceptional moat":   5,   # 총 7yr
    "Wide moat":          4,   # 총 6yr
    "Wide-Narrow moat":   3,   # 총 5yr
    "Narrow moat":        2,   # 총 4yr
    "Some moat":          1,   # 총 3yr
    "No moat":            0,   # 총 2yr
    "Unknown (fallback)": 1,
}
PHASE1_YR_BY_MOAT = {g: PHASE1_BASE_YR + e for g, e in PHASE1_EXTRA_YR.items()}

# ── Phase 1 ROE 안정장치 ─────────────────────────────────────
PHASE1_ROE_FLOOR_RATIO   = 0.80    # 과거 TTM ROE 대비 floor 비율
PHASE1_ROE_CEILING_RATIO = 2.50    # 과거 TTM ROE 대비 ceiling 비율

# ─────────────────────────────────────────────────────────────
#  ⑥ ★ v1.1 Validity 가드 (FCFF v8 Sanity Guard 와 동일 취지)
# ─────────────────────────────────────────────────────────────
RIM_SANITY_GUARD    = True    # True → 적정주가/현재가 괴리 검사 활성
RIM_SANITY_MC_RATIO = 30.0    # 허용 배율. 초과 시 ValueError → 평가 제외
                              #   (v4 기하 decay 시절 Upside 95,793% 류 차단용
                              #    2차 방어선. v5 선형 fade 로 구조적 발산은
                              #    이미 제거됐으나 입력 데이터 이상 대비 유지)

# ─────────────────────────────────────────────────────────────
#  ⑦ Re (자기자본비용) / ERP
# ─────────────────────────────────────────────────────────────
ERP_METHOD        = "damodaran_floor"  # 'damodaran_floor' | 'kospi_geo_10y'
DAMODARAN_ERP_KR  = 0.07               # Damodaran 한국 ERP
GEO_FLOOR         = 0.07               # KOSPI geo 사용 시 E(Rm) floor
RF_FALLBACK       = 0.035              # BOK API 실패 시 fallback 무위험이자율
RD_DEFAULT        = 0.045              # 부채비용 fallback (EVA WACC 산출용)
RE_FLOOR          = None    # ★ v2 옵션 A: Re clip 비활성 (FCFF 와 일관)
RE_CAP            = None    # ★ v2 옵션 A: Re clip 비활성 (FCFF 와 일관)

# ─────────────────────────────────────────────────────────────
#  ⑧ 단위 환산 / 재무 항목 키 (수정 불필요)
# ─────────────────────────────────────────────────────────────
FS_UNIT_MULTIPLIER        = 1_000        # 재무제표: 천원 → 원
MARKETCAP_UNIT_MULTIPLIER = 1_000_000    # 시가총액: 백만원 → 원

DEBT_KEYS = ["short_term_debt", "current_lt_debt", "bonds",
             "long_term_debt", "lease_liab"]
CASH_KEYS = ["cash", "short_term_invest"]


# ─────────────────────────────────────────────────────────────
#  ⑨ 체크포인트 (수정 불필요)
# ─────────────────────────────────────────────────────────────
CHECKPOINT_DIR = "../KR_FCFF_Valuation/_korea_rim_checkpoint"  # 진행 상황 기록 폴더
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
DONE_PATH = os.path.join(CHECKPOINT_DIR, "done_tickers.txt")    # 완료 종목
FAIL_PATH = os.path.join(CHECKPOINT_DIR, "failed_tickers.txt")  # 실패 종목
RANK_CSV_DIR.mkdir(parents=True, exist_ok=True)

print("[OK] 입력 변수 설정 완료")
print(f"  배치 범위        : [{TICKER_START}:{TICKER_END}]  SKIP_DONE={SKIP_DONE}"
      f"  OVERRIDE={len(RUN_TICKERS_OVERRIDE)}개")
print(f"  결과 테이블      : {TABLE_RESULT}")
print(f"  Phase2 감쇠      : LINEAR fade (v5 확정) — spread → 0, TV = 0")
print(f"  Sanity 가드      : {RIM_SANITY_GUARD}  (TP/CP {RIM_SANITY_MC_RATIO:.0f}배)")


## Cell 3 · Import & DB 연결

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Import & DB 연결 — 사용자 입력 없음 (실행만 하면 됩니다)
# ═══════════════════════════════════════════════════════════════

# ── 표준 라이브러리 ──────────────────────────────────────────────
import gc, math, time, traceback, warnings
from datetime import datetime, date, timedelta
from typing import Optional, Dict, Any, List, Tuple

# ── 외부 라이브러리 ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import pymysql
from scipy import stats
from IPython.display import display
import matplotlib
matplotlib.rcParams["axes.unicode_minus"] = False
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sqlalchemy import text

# ── 내부 모듈 (Cell 1 경로 자동 감지 덕분에 두 PC 모두 동일하게 import) ──
from DATA.config import get_db_info, get_engine
from DATA.KEYS import KEYS
from DATA.korea_valuation_helpers import (
    setup_project_path, to_dg_ticker, to_price_ticker, get_pymysql_conn,
    DG_ITEM_CODES,
    load_korea_financials_wide, load_korea_revenue_forecast,
    load_korea_marketcap_latest, load_korea_price_series, load_current_price,
    load_kospi_series, get_risk_free_rate, compute_beta_10y,
    estimate_market_return, get_universe_with_min_history,
    DataQualityReport, save_quality_report_to_db,
)
from DATA.universal_ts_forecast_function_v2 import clear_memory


def log(tag: str, msg: str):
    ts = datetime.now().strftime("%H:%M:%S")
    print(f"[{ts}][{tag}] {msg}", flush=True)


# ── DB 연결 ──────────────────────────────────────────────────────
db_info = get_db_info()
engine  = get_engine(db_info)

try:
    with engine.connect() as c:
        c.execute(text("SELECT 1"))
    log("DB", f"연결 성공 host={db_info.get('host')} port={db_info.get('port')}")
except Exception as e:
    log("DB", f"연결 실패: {e}")

print("[OK] Import 완료")
print(f"[설정] FORECAST_HORIZON={FORECAST_HORIZON}Q  ERP_METHOD={ERP_METHOD}")
print(f"[설정] GDP_GROWTH={GDP_GROWTH:.1%}  Re clip: 비활성 (FCFF 와 일관)")
print("[설정] Phase1 plateau by moat (기본 2yr + moat 연장):")
for _k, _v in PHASE1_EXTRA_YR.items():
    print(f"        {_k:<22s}: {PHASE1_BASE_YR}+{_v} = {PHASE1_BASE_YR + _v}yr")


## Cell 4 · 결과 테이블 초기화 & Universe 조회

PK = (`ticker`, `date`, `year_label`) → 같은 평가일 재실행은 갱신(UPSERT),
날짜가 다르면 시계열로 누적.

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  결과 테이블 초기화 & Universe 조회 — 사용자 입력 없음
#  - PK = (ticker, date, year_label) → Ph1/Ph2 각 연도별 행 저장
#  - 평가일(date)이 달라지면 별도 행 → 시계열 추적 가능
# ═══════════════════════════════════════════════════════════════

if True:
    CREATE_SQL = f"""
    CREATE TABLE IF NOT EXISTS `{TABLE_RESULT}` (
      `id`              BIGINT      NOT NULL AUTO_INCREMENT,
      `date`            DATE        NOT NULL COMMENT '평가 실행일',
      `ticker`          VARCHAR(20) NOT NULL,
      `year_label`      VARCHAR(10)          COMMENT '2026 / Ph2-Y3 / TV',
      `phase`           VARCHAR(10)          COMMENT 'ph1 / ph1e / ph2 / tv',
      `sales_forecast`  DOUBLE               COMMENT '연간 매출 (원)',
      `npm_forecast`    DOUBLE,
      `asset_turnover`  DOUBLE,
      `fin_leverage`    DOUBLE,
      `roe_forecast`    DOUBLE,
      `re`              DOUBLE               COMMENT 'Cost of Equity',
      `ri_spread`       DOUBLE               COMMENT 'ROE - Re',
      `bv_start`        DOUBLE               COMMENT 'BV 또는 IC (원)',
      `ri`              DOUBLE               COMMENT 'Residual Income (원)',
      `pv_ri`           DOUBLE               COMMENT 'PV of RI',
      `moat_label`      VARCHAR(30),
      `rho`             DOUBLE               COMMENT 'omega (moat 메타·v5 감쇠 미사용)',
      `n_phase2`        INT,
      `g_terminal`      DOUBLE,
      `pv_all_ri`       DOUBLE,
      `terminal_value`  DOUBLE,
      `intrinsic_value` DOUBLE,
      `current_price`   DOUBLE,
      `upside_pct`      DOUBLE,
      `target_price`    DOUBLE,
      `beta_raw`        DOUBLE,
      `beta_blume`      DOUBLE,
      `bv_source`       VARCHAR(10)          COMMENT 'Equity / IC_fallback',
      `revenue_quarters`INT,
      `forecast_model`  VARCHAR(20),
      `forecast_date`   DATE,
      `fade_mode`       VARCHAR(20)          COMMENT 'v1.1 Phase2 감쇠 방식 (linear_v5)',
      `sanity_ratio`    DOUBLE               COMMENT 'v1.1 TP/CP 배율 (Sanity 가드)',
      `spread0`         DOUBLE               COMMENT 'v1.1 Phase2 시작 스프레드',
      `created_at`      DATETIME    DEFAULT CURRENT_TIMESTAMP,
      PRIMARY KEY (`id`),
      UNIQUE KEY uq_main (`ticker`, `date`, `year_label`),
      INDEX idx_ticker (`ticker`),
      INDEX idx_date   (`date`)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
    """

    conn = get_pymysql_conn(db_info)
    try:
        with conn.cursor() as cur:
            cur.execute(CREATE_SQL)
        conn.commit()
        log("DB", f"테이블 준비 완료: {TABLE_RESULT}")
    finally:
        conn.close()

    # ★ v1.1 신규 컬럼 보강 (이미 있으면 무시)
    _ALTER_COLS = [
        ("fade_mode",    "VARCHAR(20)"),
        ("sanity_ratio", "DOUBLE"),
        ("spread0",      "DOUBLE"),
    ]
    conn = get_pymysql_conn(db_info)
    try:
        with conn.cursor() as cur:
            for _c, _t in _ALTER_COLS:
                try:
                    cur.execute(f"ALTER TABLE `{TABLE_RESULT}` ADD COLUMN `{_c}` {_t}")
                except Exception:
                    pass  # 이미 존재
        conn.commit()
    finally:
        conn.close()

# ── Universe 조회 ────────────────────────────────────────────
universe_df = get_universe_with_min_history(
    db_info, min_quarters=MIN_REVENUE_QUARTERS, require_consecutive=False)
KOREA_TICKER_LIST = universe_df["ticker"].tolist()
print(f"\n[Universe] 매출 >= {MIN_REVENUE_QUARTERS}분기 종목: {len(KOREA_TICKER_LIST):,}개")


## Cell 5 · 시장 파라미터 (Rf, KOSPI, E(Rm)) — 1회 계산 후 캐싱

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  시장 공통 파라미터 (Rf, KOSPI, E(Rm)) — 사용자 입력 없음
# ═══════════════════════════════════════════════════════════════

# 1. Risk-free (BOK 10Y 국고채)
RF, RF_SOURCE = get_risk_free_rate(KEYS["BOK"], fallback_rate=RF_FALLBACK)
log("MKT", f"Rf = {RF:.4%}  (source: {RF_SOURCE})")

# 2. KOSPI (helpers 의 3단 fallback 로더: FDR → pykrx → yfinance)
KOSPI_PX = load_kospi_series(
    start_date=(datetime.today() - timedelta(days=365 * 12)).strftime("%Y-%m-%d"))
log("MKT", f"KOSPI {len(KOSPI_PX):,}거래일  "
           f"({KOSPI_PX.index.min().date()} ~ {KOSPI_PX.index.max().date()})")

# 3. E(Rm)
mkt = estimate_market_return(
    method=ERP_METHOD, rf=RF, kospi_series=KOSPI_PX,
    years=10, damodaran_erp_kr=DAMODARAN_ERP_KR, geo_floor=GEO_FLOOR)
E_RM = mkt["e_rm"]
ERP  = mkt["erp"]
log("MKT", f"E(Rm) = {E_RM:.4%}  ERP = {ERP:.4%}  ({mkt['note']})")

print()
print("=" * 60)
print(f"  Rf      = {RF:>7.3%}")
print(f"  ERP     = {ERP:>7.3%}")
print(f"  E(Rm)   = {E_RM:>7.3%}")
print(f"  Re clip = 비활성 (FCFF 와 일관 — raw Re 사용)")
print("=" * 60)


## Cell 6 · `load_korea_revenue_forecast` 패치

> ⚠️ Cell 3(import 셀)을 다시 실행하면 패치가 풀리므로, 이 셀도 반드시 재실행하세요.

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  PATCH 셀 · load_korea_revenue_forecast 교체
#  (구스키마 created_at 버그 → 매출 forecast 상수 복제 문제 수정)
#
#  ▸ 붙여넣을 위치 : korea_fcff_dcf_valuation_v7.ipynb 의
#                    import 셀(DATA.korea_valuation_helpers import) "바로 다음".
#                    KoreaDCFModel 실행(Cell 9/10) 전이면 어디든 OK.
#  ▸ 원리          : 노트북 전역의 load_korea_revenue_forecast 이름을
#                    아래 수정판으로 재바인딩 → KoreaDCFModel.load_sales 가
#                    호출 시점에 이 패치판을 사용.
#  ▸ 주의          : import 셀을 다시 실행하면 패치가 풀리므로,
#                    import 셀 재실행 후에는 이 셀도 재실행할 것.
#
#  ▸ 수정 내용 (korea_revenue_analysis_notebook_v3 의 fetch_forecast 와 동일 원리)
#    1) 버전 식별: MAX(created_at) → MAX(updated_at) (자동 감지, 없으면 created_at)
#       - 구스키마 UNIQUE KEY(date,ticker,indicator) 에서는 재예측 시
#         겹치는 분기의 created_at 이 갱신되지 않아 '새로 생긴 분기 1개'만
#         최신 버전으로 잡히는 문제가 있었음 (예: A031330 → 2028Q1 한 분기).
#    2) 같은 분기 중복 시 가장 최근 갱신분만 사용.
#    3) 검증 추가 — 아래 셋 중 하나라도 걸리면 ValueError 로 즉시 차단
#       (잘못된 valuation 이 DB 에 저장되는 것을 막기 위함):
#       a. 분기 수 < horizon
#       b. 분기 불연속 (중간 분기 누락)
#       c. 전 분기 동일값 (상수 시계열 — 이번 버그의 증상)
# ═══════════════════════════════════════════════════════════════
from sqlalchemy import text as _sa_text

_FC_MODEL_PRIORITY = ("Ensemble", "SARIMA", "ETS", "Theta")
_FC_RECENCY_CACHE: dict = {}


def _fc_recency_col(table_name: str) -> str:
    """updated_at 컬럼이 있으면 그것을, 없으면 created_at 을 버전 기준으로 사용."""
    if table_name in _FC_RECENCY_CACHE:
        return _FC_RECENCY_CACHE[table_name]
    sql = ("SELECT COLUMN_NAME FROM information_schema.COLUMNS "
           "WHERE TABLE_SCHEMA = :db AND TABLE_NAME = :tbl")
    with engine.connect() as conn:
        cols = {r[0].lower() for r in conn.execute(
            _sa_text(sql), {"db": db_info["database"], "tbl": table_name}
        ).fetchall()}
    rc = "updated_at" if "updated_at" in cols else "created_at"
    _FC_RECENCY_CACHE[table_name] = rc
    if rc == "created_at":
        log("PATCH", f"[주의] {table_name} 에 updated_at 없음 → created_at 기준. "
                     "구스키마면 일부 종목 분기 누락 가능 — updated_at 추가 권장.")
    return rc


def load_korea_revenue_forecast(ticker: str,
                                db_info: dict,
                                table_name: str = "korea_revenue_forecast_result",
                                horizon: int = 8):
    """[PATCHED v7-fix] 최신 '실행 버전' 전체 분기를 안전하게 로드.

    Returns
    -------
    (pd.Series, str, str)
        forecast : 천원 단위, index=분기 date (호출측에서 ×1000 변환)
        model    : 사용된 indicator (Ensemble 우선)
        run_date : 예측 실행 버전 날짜 (YYYY-MM-DD)
    """
    rc = _fc_recency_col(table_name)

    df, used_model, run_date = None, "", None
    for model in _FC_MODEL_PRIORITY:
        # indicator 가 'Ensemble' 형태든 '매출액(천원)_Ensemble' 형태든 모두 매칭
        with engine.connect() as conn:
            row = conn.execute(_sa_text(
                f"SELECT MAX(DATE({rc})) FROM {table_name} "
                f"WHERE ticker = :t AND (indicator = :m OR indicator LIKE :ml)"
            ), {"t": ticker, "m": model, "ml": f"%\\_{model}"}).fetchone()
        if row is None or row[0] is None:
            continue
        run_date = str(row[0])
        cand = pd.read_sql(_sa_text(
            f"SELECT date, value, {rc} AS run_ts FROM {table_name} "
            f"WHERE ticker = :t AND (indicator = :m OR indicator LIKE :ml) "
            f"  AND DATE({rc}) = :fd "
            f"ORDER BY date"
        ), engine, params={"t": ticker, "m": model,
                           "ml": f"%\\_{model}", "fd": run_date})
        if not cand.empty:
            df, used_model = cand, model
            break

    if df is None:
        return pd.Series(dtype=float), "", None

    df["date"]   = pd.to_datetime(df["date"])
    df["run_ts"] = pd.to_datetime(df["run_ts"])
    df = (df.sort_values(["date", "run_ts"])
            .drop_duplicates(subset=["date"], keep="last"))
    fc = df.set_index("date")["value"].astype(float).sort_index()

    # ── 검증 a: 분기 수 ───────────────────────────────────────
    if len(fc) < horizon:
        raise ValueError(
            f"[{ticker}] forecast 분기 {len(fc)}개 < horizon {horizon} "
            f"(version={rc} {run_date}, indicator={used_model}). "
            f"구스키마 created_at 잔존 가능성 — 해당 종목 재예측 또는 "
            f"테이블 updated_at 마이그레이션 필요."
        )
    fc = fc.iloc[:horizon]

    # ── 검증 b: 분기 연속성 ──────────────────────────────────
    per = fc.index.to_period("Q")
    gaps = (per[1:].astype("int64") - per[:-1].astype("int64"))
    if (gaps != 1).any():
        bad = [str(per[i + 1]) for i, g in enumerate(gaps) if g != 1]
        raise ValueError(
            f"[{ticker}] forecast 분기 불연속: {bad} "
            f"(version={rc} {run_date}). 해당 종목 재예측 필요."
        )

    # ── 검증 c: 상수 시계열 (이번 버그의 증상) ────────────────
    if fc.nunique() == 1:
        raise ValueError(
            f"[{ticker}] forecast {horizon}개 분기가 모두 동일값"
            f"({fc.iloc[0]:,.0f}천원) — 단일 분기 복제 의심. "
            f"(version={rc} {run_date}). 해당 종목 재예측 필요."
        )

    return fc, used_model, run_date


# log("PATCH", "load_korea_revenue_forecast → updated_at 버전 선택 + 3중 검증 패치 적용 완료")


## Cell 7 · `KoreaRIMModel` v1.1 클래스 & `process_one_ticker_rim_kr`

★ v5 선형 fade 내장 · Moat 등급표 외부화 · Sanity 가드 포함.
클래스 정의만 수행하며 **평가는 실행하지 않습니다**.

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  KoreaRIMModel v1.1 — Ohlson (1995) Residual Income Valuation
#  ─────────────────────────────────────────────────────────────
#  v5 → v1.1 변경 (★ 이중 평가 제거 / 단일화)
#    ① compute_ri_path 를 v5 선형 fade 버전으로 '클래스 본문에 확정'
#       - 구 노트북: 클래스에 v4 기하 decay(RI_t = ω x RI_{t-1})가 들어있고
#         별도 패치 셀이 런타임에 v5 로 교체하는 구조.
#         클래스 셀 말미의 TEST_TICKER 실행이 패치 '이전'에 돌아
#         같은 종목이 v4(ω=0.980) 로 한 번, 패치 후 진단 셀에서 v5 로
#         또 한 번 평가되어 적정주가가 2개 나왔음.
#         (A204620: v4 TP=6,845원 / v5 TP=7,540원)
#       - v1.1: 패치 셀 자체를 없애고 v5 를 본문에 확정 → 실행 순서와
#         무관하게 항상 동일한 결과.
#    ② 클래스 셀 내부의 단일 종목 테스트/plot 실행 제거
#       (평가는 실행 셀에서 단 한 번만)
#    ③ Sanity 가드 추가 : |TP/CP| > RIM_SANITY_MC_RATIO 이면 평가 제외
#    ④ omega 는 moat 등급 메타데이터로만 유지 (Phase 2 감쇠에 미사용)
# ═══════════════════════════════════════════════════════════════

class KoreaRIMModel:
    """
    한국 주식 RIM 평가 모델.

    공식:
        IV = BV_0 + Σ RI_t / (1+Re)^t + PV(TV)
        RI_t = (ROE_t - Re) × BV_{t-1}
        BV_t = BV_{t-1} × (1 + ROE_t × retention)  [with Re floor]

    Phase 1: DuPont 기반 ROE forecast (2yr 실제 + moat plateau 확장)
    Phase 2: AR(1) decay  RI(t) = ω × RI(t-1)   [5~30yr moat 등급별]
    Phase 3: TV = RI_last × (1+g) / (Re - g)

    US v11과 동일한 6-Tier moat 분류 + OR 로직 + n_pos 페널티.
    """

    def __init__(self, ticker, engine, db_info, rf, e_rm, kospi_series,
                 forecast_horizon=FORECAST_HORIZON,
                 min_history=MIN_HISTORY,
                 gdp_growth=GDP_GROWTH,
                 verbose=False):
        self.ticker_dg    = to_dg_ticker(ticker)
        self.ticker_price = to_price_ticker(ticker)
        self.engine       = engine
        self.db_info      = db_info
        self.rf           = rf
        self.e_rm         = e_rm
        self.erp          = e_rm - rf
        self.kospi        = kospi_series
        self.horizon      = forecast_horizon
        self.min_history  = min_history
        self.gdp_growth   = gdp_growth
        self.verbose      = verbose

        self._sales_actual   = None
        self._sales_forecast = None
        self._used_model     = ""
        self._forecast_date  = None
        self._fs_wide        = None
        self._re             = None
        self._eva_cache      = None
        self._beta_info      = None

        self._using_ic    = False
        self._n_phase1    = 2
        self._bv_source   = "Equity"
        self.result_df    = None
        self.valuation    = None

        self.report = DataQualityReport(ticker=self.ticker_dg)

    # ─────────────────────────────────────────────────────────
    # Utility
    # ─────────────────────────────────────────────────────────
    @staticmethod
    def _winsorize(s, limits=WINSORIZE_LIMITS):
        s = s.dropna()
        if len(s) < 4: return s
        lo, hi = s.quantile(limits[0]), s.quantile(limits[1])
        return s.clip(lo, hi)

    @staticmethod
    def _ols_ratio(x, y):
        mask = x.notna() & y.notna() & (x != 0)
        if mask.sum() < OLS_MIN_SAMPLES:
            return np.nan, -1.0, int(mask.sum())
        slope, _, r, _, _ = stats.linregress(x[mask], y[mask])
        return float(slope), float(r**2), int(mask.sum())

    @staticmethod
    def _n(val):
        if val is None: return None
        try:
            f = float(val)
            return None if (np.isnan(f) or np.isinf(f)) else f
        except (TypeError, ValueError):
            return val

    # ─────────────────────────────────────────────────────────
    # 1. Data Loading
    # ─────────────────────────────────────────────────────────
    def load_sales(self):
        wide = load_korea_financials_wide(
            self.ticker_dg, self.db_info, table_name=TABLE_FS,
            item_keys=["revenue"], fillna_zero=False)
        actual = wide["revenue"].dropna() * FS_UNIT_MULTIPLIER
        if actual.empty or len(actual) < MIN_REVENUE_QUARTERS:
            self.report.add("revenue_actual", "missing", n_obs=len(actual),
                            note=f"actual {len(actual)}Q < {MIN_REVENUE_QUARTERS}")
            raise ValueError(
                f"[{self.ticker_dg}] 매출 actual 부족 ({len(actual)}Q < {MIN_REVENUE_QUARTERS}Q)")
        self.report.add("revenue_actual", "ok", n_obs=len(actual))

        forecast, model_name, ca = load_korea_revenue_forecast(
            self.ticker_dg, self.db_info, table_name=TABLE_FORECAST,
            horizon=self.horizon)
        if forecast.empty:
            self.report.add("revenue_forecast", "missing", n_obs=0,
                            note="korea_revenue_forecast_result 없음")
            raise ValueError(f"[{self.ticker_dg}] 매출 forecast 없음")
        forecast = forecast * FS_UNIT_MULTIPLIER
        self.report.add("revenue_forecast", "ok", n_obs=len(forecast),
                        note=f"model={model_name}")

        self._sales_actual   = actual
        self._sales_forecast = forecast.iloc[:self.horizon]
        self._used_model     = model_name
        self._forecast_date  = ca

        if self.verbose:
            log(self.ticker_dg,
                f"Sales actual={len(actual)}Q forecast={len(self._sales_forecast)}Q "
                f"({model_name})")
        return self

    def load_financials(self):
        wide = load_korea_financials_wide(
            self.ticker_dg, self.db_info, table_name=TABLE_FS,
            item_keys=None, fillna_zero=False)
        if wide.empty:
            raise ValueError(f"[{self.ticker_dg}] FS 데이터 없음")

        non_share = [c for c in wide.columns
                     if c not in ("shares_treasury_adj", "shares_common")]
        wide[non_share] = wide[non_share] * FS_UNIT_MULTIPLIER

        # 핵심항목 검증
        missing_core = []
        if "revenue" not in wide.columns or wide["revenue"].dropna().empty:
            missing_core.append("revenue")
        if "net_income" not in wide.columns or wide["net_income"].dropna().empty:
            missing_core.append("net_income")
        if "total_equity" not in wide.columns or wide["total_equity"].dropna().empty:
            missing_core.append("total_equity")
        if missing_core:
            for k in missing_core:
                self.report.add(k, "missing", n_obs=0)
            raise ValueError(f"[{self.ticker_dg}] 핵심 항목 결측: {missing_core}")

        self.report.add("fs_core", "ok",
                        n_obs=int(wide[["revenue","net_income","total_equity"]]
                                  .notna().all(axis=1).sum()))
        self._fs_wide = wide
        if self.verbose:
            log(self.ticker_dg, f"FS wide shape={wide.shape}")
        return self

    # ─────────────────────────────────────────────────────────
    # 2. Re (Cost of Equity) — 자체 베타 10y + CAPM
    # ─────────────────────────────────────────────────────────
    def load_re(self):
        """β_blume = 0.67×|β| + 0.33  →  Re = Rf + β_blume × ERP

        ★ v2 옵션 A: clip(Re, RE_FLOOR, RE_CAP) 제거 — FCFF 모형과 일관.
           Beta·ERP 가 raw 그대로 Re 에 반영됨.
        """
        info = compute_beta_10y(
            self.ticker_dg, self.db_info, kospi_series=self.kospi,
            years=10, min_obs=750)
        self._beta_info = info

        if np.isnan(info["beta_raw"]):
            beta_blume = 1.0
            self.report.add("beta", "fallback_median", n_obs=info["n_obs"],
                            value=beta_blume, note="베타 계산 실패 → 1.0")
        else:
            beta_blume = info["beta_blume"]
            self.report.add("beta", "ok", n_obs=info["n_obs"],
                            value=beta_blume, r2=info["r_squared"],
                            note=f"β_raw={info['beta_raw']:.3f}")

        # ★ v2 옵션 A: Re clip 제거 (FCFF 와 동일하게 raw 사용)
        re = float(self.rf + beta_blume * self.erp)
        self._re = re

        if self.verbose:
            log(self.ticker_dg,
                f"Re={re:.4%}  β_raw={info.get('beta_raw', np.nan):.3f} "
                f"β_blume={beta_blume:.3f}  Rf={self.rf:.3%} ERP={self.erp:.3%}")
        return self

    # ─────────────────────────────────────────────────────────
    # 3. 보조 함수: IC / TTM ROE / Retention / Tax
    # ─────────────────────────────────────────────────────────
    def _estimate_ic(self):
        """
        Invested Capital = Equity + Total Debt
        음수 BV 기업(적자 누적 또는 대량 자사주매입) 처리용.
        모든 부채가 없으면 totalAssets × 0.4 fallback.
        """
        wide = self._fs_wide.sort_index()
        last = wide.iloc[-1]

        eq = last.get("total_equity", 0) or 0
        td = 0.0
        for k in DEBT_KEYS:
            if k in wide.columns:
                v = last.get(k, 0)
                if pd.notna(v):
                    td += float(v)
        ic = float(eq) + td
        if ic <= 0:
            ta = last.get("total_assets", 0)
            ic = float(ta) * 0.40 if pd.notna(ta) and ta > 0 else 1e11
        return max(ic, 1e10)  # 최소 100억

    def _get_historical_roe_ttm(self):
        """최근 TTM ROE 계산 (DuPont blend용)"""
        try:
            wide = self._fs_wide.sort_index()
            if "net_income" not in wide.columns or "total_equity" not in wide.columns:
                return np.nan
            df = wide[["net_income", "total_equity"]].dropna()
            df = df[df["total_equity"] > 0]
            if len(df) < 4:
                return np.nan
            ni_ttm = float(df["net_income"].iloc[-4:].sum())
            eq_avg = float(df["total_equity"].iloc[-4:].mean())
            return ni_ttm / eq_avg if eq_avg > 0 else np.nan
        except Exception:
            return np.nan

    def estimate_tax_rate(self):
        """실효세율 (한국 법정세율 22% fallback) — FCFF 모형과 일관 (★ v2 옵션 A)."""
        wide = self._fs_wide
        if "pretax_income" not in wide.columns or "tax_expense" not in wide.columns:
            self.report.add("tax_rate", "fallback_zero", n_obs=0,
                            value=0.22, note="컬럼 누락 → 법정세율 22%")
            return 0.22
        df = wide[["pretax_income", "tax_expense"]].dropna()
        df = df[df["pretax_income"] > 0]
        if df.empty:
            self.report.add("tax_rate", "fallback_zero", n_obs=0,
                            value=0.22, note="법정세율 22% fallback")
            return 0.22
        rates = (df["tax_expense"] / df["pretax_income"]).clip(0, 0.40)
        med = float(rates.median())
        self.report.add("tax_rate", "ok", n_obs=len(rates), value=med)
        return med

    def estimate_retention(self):
        # ★ v1.1: run() 한 번에 forecast_roe_phase1 / compute_ri_path 에서
        #         두 번 호출되므로 결과를 캐싱 (동일 값 · 중복 로그 방지)
        if getattr(self, '_retention_cache', None) is not None:
            return self._retention_cache
        self._retention_cache = self._estimate_retention_impl()
        return self._retention_cache

    def _estimate_retention_impl(self):
        """
        실질 이익유보율 = 1 - 배당성향.
        한국 기업은 buyback 데이터 신뢰성 낮음 → 배당(dividends_paid)만 사용.
        Clean Surplus 역산 fallback (NI - ΔEquity).
        """
        wide = self._fs_wide.sort_index()
        if "net_income" not in wide.columns:
            self.report.add("retention", "fallback_zero", value=0.70,
                            note="net_income 없음")
            return 0.70
        ni = wide["net_income"]
        if (ni > 0).sum() == 0:
            self.report.add("retention", "fallback_zero", value=0.70,
                            note="NI 양수 분기 0")
            return 0.70

        div = None
        # 방법 1: dividends_paid 직접 사용
        if "dividends_paid" in wide.columns:
            d = wide["dividends_paid"].abs()
            if (d > 0).sum() >= 4:
                div = d

        # 방법 2: Clean Surplus 역산 (NI - ΔEquity)
        if div is None and "total_equity" in wide.columns:
            eq = wide["total_equity"]
            delta_eq = eq.diff()
            implied = (ni - delta_eq).clip(lower=0)
            if (implied > 0).sum() >= 4:
                div = implied

        if div is None:
            self.report.add("retention", "fallback_zero", value=0.70,
                            note="배당/clean surplus 모두 실패")
            return 0.70

        payout = (div / ni.abs()).replace([np.inf, -np.inf], np.nan)
        po = payout.iloc[-8:].dropna()
        if po.empty:
            po = payout.dropna()
        if po.empty:
            self.report.add("retention", "fallback_zero", value=0.70)
            return 0.70

        payout_med = float(po.clip(0, 3.0).median())
        retention = float(np.clip(1.0 - payout_med, RETENTION_FLOOR, 0.98))

        self.report.add("retention", "ok", n_obs=len(po), value=retention,
                        note=f"payout_med={payout_med:.2%}")
        if self.verbose:
            log(self.ticker_dg, f"Retention={retention:.3f}  payout={payout_med:.3f}")
        return retention

    # ─────────────────────────────────────────────────────────
    # 4. DuPont 계수 추정
    # ─────────────────────────────────────────────────────────
    def estimate_dupont_coefs(self):
        """NPM (OLS → median), AT (TTM), FL (median)"""
        wide = self._fs_wide

        # ── NPM = NetIncome / Revenue (OLS slope) ────────────
        df_npm = wide[["revenue", "net_income"]].dropna()
        df_npm = df_npm[df_npm["revenue"] > 0]
        npm_series = self._winsorize((df_npm["net_income"] / df_npm["revenue"]).dropna())
        npm_median = float(npm_series.median()) if not npm_series.empty else 0.05
        npm_coef, npm_method = npm_median, "median"
        r2_npm = -1.0

        if len(df_npm) >= OLS_MIN_SAMPLES:
            slope, r2_npm, n = self._ols_ratio(df_npm["revenue"], df_npm["net_income"])
            if not np.isnan(slope) and r2_npm >= OLS_MIN_R2:
                npm_coef, npm_method = float(slope), "ols"
                self.report.add("npm", "ok", n_obs=n, value=npm_coef,
                                r2=r2_npm, note="OLS slope")
            else:
                self.report.add("npm", "fallback_median", n_obs=n,
                                value=npm_median, r2=r2_npm,
                                note=f"R²={r2_npm:.2f} → median")
        else:
            self.report.add("npm", "fallback_median", n_obs=len(df_npm),
                            value=npm_median,
                            note=f"n={len(df_npm)} < {OLS_MIN_SAMPLES}")

        # ── AT = TTM Sales / TotalAssets (median) ────────────
        at_median = 0.70
        if "total_assets" in wide.columns:
            df_at = wide[["revenue", "total_assets"]].dropna().sort_index()
            df_at = df_at[df_at["total_assets"] > 0]
            if len(df_at) >= 4:
                df_at["ttm"] = df_at["revenue"].rolling(4).sum()
                atm = df_at.dropna(subset=["ttm"])
                ratios = self._winsorize((atm["ttm"] / atm["total_assets"]).dropna())
                if not ratios.empty:
                    at_median = float(ratios.median())
                    self.report.add("asset_turnover", "ok",
                                    n_obs=len(ratios), value=at_median)
                else:
                    self.report.add("asset_turnover", "fallback_zero", value=at_median)
            else:
                self.report.add("asset_turnover", "fallback_zero", value=at_median,
                                note="total_assets < 4Q")
        else:
            self.report.add("asset_turnover", "fallback_zero", value=at_median,
                            note="total_assets 컬럼 없음")

        # ── FL = TotalAssets / Equity (median, clip [1, 20]) ──
        fl_median = 2.5
        if "total_assets" in wide.columns and "total_equity" in wide.columns:
            df_fl = wide[["total_assets", "total_equity"]].dropna()
            df_fl = df_fl[(df_fl["total_assets"] > 0) & (df_fl["total_equity"] > 0)]
            if not df_fl.empty:
                ratios = self._winsorize(
                    (df_fl["total_assets"] / df_fl["total_equity"]).dropna())
                fl_median = float(np.clip(ratios.median(), 1.0, 20.0))
                self.report.add("financial_leverage", "ok",
                                n_obs=len(ratios), value=fl_median)
            else:
                self.report.add("financial_leverage", "fallback_zero", value=fl_median,
                                note="equity>0 분기 없음")
        else:
            self.report.add("financial_leverage", "fallback_zero", value=fl_median)

        if self.verbose:
            log(self.ticker_dg,
                f"DuPont  NPM={npm_coef:.4f}({npm_method})  "
                f"AT={at_median:.3f}(TTM)  FL={fl_median:.2f}")

        return {
            "npm_coef":   npm_coef,
            "npm_method": npm_method,
            "npm_r2":     r2_npm,
            "at_median":  at_median,
            "fl_median":  fl_median,
        }

    # ─────────────────────────────────────────────────────────
    # 5. Phase 1 ROE forecast
    # ─────────────────────────────────────────────────────────
    def forecast_roe_phase1(self, coefs, n_years=2):
        """
        Phase 1 = 실제 매출 예측 구간 (forecast) + moat plateau 연장 구간 (gdp_ext).
        연장 구간에서는 Forecast 마지막 ROE를 고정 유지 (Mauboussin CAP 반영).
        """
        fc_q = self._sales_forecast
        re = self._re

        # Sales 연간 집계 (실제 예측)
        annual_sales_raw = []
        for yr in range(0, len(fc_q), 4):
            chunk = fc_q.iloc[yr: yr + 4]
            annual_sales_raw.append({
                "year": fc_q.index[yr].year,
                "sales": float(chunk.sum()),
                "source": "forecast",
            })

        last_yr = annual_sales_raw[-1]["year"] if annual_sales_raw else datetime.now().year
        last_sal = annual_sales_raw[-1]["sales"] if annual_sales_raw else 0.0

        # Plateau 연장 (GDP 복리)
        annual_sales = list(annual_sales_raw)
        for i in range(len(annual_sales_raw), n_years):
            extra = i - len(annual_sales_raw) + 1
            annual_sales.append({
                "year": last_yr + extra,
                "sales": last_sal * (1 + GDP_GROWTH) ** extra,
                "source": "gdp_ext",
            })

        # BV₀ 확보 — 음수면 IC로 대체
        wide = self._fs_wide.sort_index()
        eq_ser = wide["total_equity"].dropna()
        bv0_raw = float(eq_ser.iloc[-1]) if not eq_ser.empty else -1.0
        if bv0_raw <= 0:
            bv0 = self._estimate_ic()
            self._using_ic = True
            self._bv_source = "IC_fallback"
            self.report.add("bv0", "fallback_median", n_obs=0, value=bv0,
                            note=f"음수 BV({bv0_raw/1e12:.2f}조) → IC 대체")
            if self.verbose:
                log(self.ticker_dg,
                    f"BV 음수 → IC={bv0/1e12:.2f}조 사용")
        else:
            bv0 = bv0_raw
            self._using_ic = False
            self._bv_source = "Equity"
            self.report.add("bv0", "ok", value=bv0)

        retention = self.estimate_retention()
        hist_roe = self._get_historical_roe_ttm()

        rows = []
        bv_start = bv0
        roe_fixed = None

        for yr_info in annual_sales:
            is_flat = (yr_info.get("source", "forecast") == "gdp_ext")

            if not is_flat:
                sales = yr_info["sales"]
                ni = coefs["npm_coef"] * sales
                npm = ni / sales if sales > 0 else 0.0
                at_est = coefs["at_median"]
                fl_est = coefs["fl_median"]
                roe_raw = npm * at_est * fl_est

                if not np.isnan(hist_roe) and hist_roe > 0:
                    # blend 65/35 + floor/ceiling
                    roe_blend = 0.65 * roe_raw + 0.35 * hist_roe
                    floor_roe = hist_roe * PHASE1_ROE_FLOOR_RATIO
                    ceil_roe  = hist_roe * PHASE1_ROE_CEILING_RATIO
                    if roe_raw < floor_roe:
                        roe = max(roe_blend, floor_roe)
                    elif roe_raw > ceil_roe:
                        roe = min(roe_blend, hist_roe * 1.50)
                    else:
                        roe = roe_blend
                    roe = float(np.clip(roe, -0.99, 3.0))
                else:
                    roe = float(np.clip(roe_raw, -0.99, 2.5))

                roe_fixed = roe
            else:
                roe = roe_fixed if roe_fixed is not None else (hist_roe or re + 0.05)
                sales = yr_info["sales"]
                ni = coefs["npm_coef"] * sales
                npm = ni / sales if sales > 0 else 0.0
                at_est = coefs["at_median"]
                fl_est = coefs["fl_median"]

            ri_spread = roe - re
            ri = ri_spread * bv_start

            rows.append({
                "year":         yr_info["year"],
                "phase":        "ph1" if not is_flat else "ph1e",
                "sales_annual": sales,
                "net_income":   ni,
                "npm":          npm,
                "at":           at_est,
                "fl":           fl_est,
                "roe":          roe,
                "re":           re,
                "ri_spread":    ri_spread,
                "bv_start":     bv_start,
                "ri":           ri,
            })
            # BV 진행: BV × (1 + Re × b_bv) + RI × b_bv
            b_bv = retention if retention < 0 else min(retention, BV_RETENTION_CAP)
            bv_start = bv_start * (1.0 + re * b_bv) + ri * b_bv

        return pd.DataFrame(rows), bv_start

    # ─────────────────────────────────────────────────────────
    # 6. EVA spread + Moat 분류 (6-Tier)
    # ─────────────────────────────────────────────────────────
    def compute_wacc_simple(self):
        """EVA 계산용 간단 WACC"""
        re = self._re or 0.10
        tax = self.estimate_tax_rate()
        wide = self._fs_wide

        # Rd
        rd = RD_DEFAULT
        if "interest_expense" in wide.columns:
            total_debt = pd.Series(0.0, index=wide.index)
            for k in DEBT_KEYS:
                if k in wide.columns:
                    total_debt = total_debt + wide[k].fillna(0)
            td_avg = (total_debt + total_debt.shift(1)) / 2
            ie = wide["interest_expense"].abs()
            valid = (td_avg > 0) & ie.notna()
            if valid.sum() >= 2:
                rd = float((ie[valid] / td_avg[valid]).clip(0, 0.20).median())

        # 시가총액
        mc, _ = load_korea_marketcap_latest(
            self.ticker_dg, self.db_info, table_name=TABLE_MARKETCAP)
        if mc is None or mc <= 0:
            return re  # fallback: Re만 사용
        mkt_cap = mc * MARKETCAP_UNIT_MULTIPLIER

        # 총부채 (latest)
        total_debt_latest = 0.0
        for k in DEBT_KEYS:
            if k in wide.columns:
                v = wide[k].dropna()
                if not v.empty:
                    total_debt_latest += float(v.iloc[-1])

        V = mkt_cap + total_debt_latest
        if V <= 0:
            return re
        wacc = re * (mkt_cap/V) + rd * (1-tax) * (total_debt_latest/V)
        return float(np.clip(wacc, 0.04, 0.25))

    def compute_eva_spread(self):
        """EVA spread = ROIC_TTM - WACC, plus ROE-Re spread"""
        wacc = self.compute_wacc_simple()
        tax = self.estimate_tax_rate()
        wide = self._fs_wide

        if "operating_income" not in wide.columns:
            return {"roic": np.nan, "wacc": wacc, "eva_spread": np.nan,
                    "roe_re_spread": np.nan, "n_positive": 0, "eva_series": []}

        # NOPAT quarterly
        nopat_q = wide["operating_income"] * (1 - tax)

        # IC = Equity + Total Debt - Cash
        td_s = pd.Series(0.0, index=wide.index)
        for k in DEBT_KEYS:
            if k in wide.columns:
                td_s = td_s + wide[k].fillna(0)
        cash_s = pd.Series(0.0, index=wide.index)
        for k in CASH_KEYS:
            if k in wide.columns:
                cash_s = cash_s + wide[k].fillna(0)
        ic_s = wide["total_equity"].fillna(0) + td_s - cash_s

        merged = pd.concat([nopat_q.rename("nopat_q"), ic_s.rename("ic")],
                           axis=1, join="inner").dropna().sort_index()
        if len(merged) < 4:
            return {"roic": np.nan, "wacc": wacc, "eva_spread": np.nan,
                    "roe_re_spread": np.nan, "n_positive": 0, "eva_series": []}

        eva_series = []
        dates = sorted(merged.index)
        for i, dt in enumerate(dates):
            if i < 3: continue
            nopat_ttm = float(merged["nopat_q"].iloc[i-3:i+1].sum())
            ic_snap = float(merged["ic"].iloc[i])
            if ic_snap <= 0: continue
            roic_q = nopat_ttm / ic_snap
            eva_q = roic_q - wacc
            eva_series.append({"date": dt, "roic": roic_q, "eva_spread": eva_q})

        if not eva_series:
            return {"roic": np.nan, "wacc": wacc, "eva_spread": np.nan,
                    "roe_re_spread": np.nan, "n_positive": 0, "eva_series": []}

        latest = eva_series[-1]
        recent_20 = eva_series[-20:]
        n_pos = sum(1 for e in recent_20 if e["eva_spread"] > 0)

        # ROE - Re spread
        re = self._re or 0.10
        roe_re_spread = np.nan
        df_re = wide[["net_income", "total_equity"]].dropna()
        df_re = df_re[df_re["total_equity"] > 0]
        if len(df_re) >= 4:
            ni_ttm = float(df_re["net_income"].iloc[-4:].sum())
            eq_avg = float(df_re["total_equity"].iloc[-4:].mean())
            if eq_avg > 0:
                roe_ttm = ni_ttm / eq_avg
                roe_re_spread = float(roe_ttm - re)

        result = {
            "roic":          latest["roic"],
            "wacc":          wacc,
            "eva_spread":    latest["eva_spread"],
            "roe_re_spread": roe_re_spread,
            "n_positive":    n_pos,
            "eva_series":    eva_series,
        }
        self._eva_cache = result

        if self.verbose:
            rrs_str = f"{roe_re_spread*100:+.1f}%" if not np.isnan(roe_re_spread) else "N/A"
            log(self.ticker_dg,
                f"EVA: ROIC={latest['roic']:.2%} WACC={wacc:.2%} "
                f"spread={latest['eva_spread']:+.2%} n_pos={n_pos}/20  ROE-Re={rrs_str}")
        return result

    def moat_to_omega_and_years(self, eva):
        """6-Tier moat 분류 → (omega, n_phase2, grade, n_phase1).

        ★ v1.1: 등급표를 클래스 내부 하드코딩에서 입력 변수 셀의
                MOAT_TIERS / PHASE1_EXTRA_YR 로 이관 (Cell 2 에서 조정 가능).
        ※ omega 는 moat 등급의 메타데이터로만 반환되며, v5 선형 fade 에서는
          Phase 2 감쇠 계산에 사용되지 않습니다 (Phase 2 년수 n 만 사용).
        """
        eva_spread    = eva.get("eva_spread",    np.nan)
        roe_re_spread = eva.get("roe_re_spread", np.nan)
        n_pos         = eva.get("n_positive", 0)

        if np.isnan(eva_spread) and np.isnan(roe_re_spread):
            g = "Unknown (fallback)"
            return (MOAT_UNKNOWN_OMEGA, MOAT_UNKNOWN_PHASE2, g,
                    PHASE1_YR_BY_MOAT.get(g, 3))

        eva_s = eva_spread    if not np.isnan(eva_spread)    else -np.inf
        roe_s = roe_re_spread if not np.isnan(roe_re_spread) else -np.inf

        # ── 1차 OR 분류 (MOAT_TIERS 는 입력 변수 셀에서 정의) ──
        grade, omega, n = MOAT_TIERS[-1][0], MOAT_TIERS[-1][1], MOAT_TIERS[-1][2]
        for label, w, yr, eva_th, roe_th, _np_min in MOAT_TIERS[:-1]:
            if eva_s > eva_th or roe_s > roe_th:
                grade, omega, n = label, w, yr
                break

        # ── n_pos 페널티: 실적 일관성 부족 시 한 등급 강등 ──
        _order    = [t[0] for t in MOAT_TIERS]
        _params   = {t[0]: (t[1], t[2]) for t in MOAT_TIERS}
        _npos_min = {t[0]: t[5] for t in MOAT_TIERS}
        if n_pos < _npos_min.get(grade, 0):
            idx = _order.index(grade)
            if idx < len(_order) - 1:
                grade = _order[idx + 1]
                omega, n = _params[grade]

        n_phase1 = PHASE1_YR_BY_MOAT.get(grade, 2)
        return omega, n, grade, n_phase1

    # ─────────────────────────────────────────────────────────
    # 7. RI Path + Valuation
    # ─────────────────────────────────────────────────────────
    def compute_ri_path(self):
        """Phase 1 (DuPont) + Phase 2 (★ v5 LINEAR fade) RI 경로 생성.

        ─────────────────────────────────────────────────────────
        ★ v1.1: 구 v5 노트북의 '패치 셀'(compute_ri_path 를 런타임에 교체)
                내용을 클래스 본문에 그대로 흡수했습니다.
                → 클래스 셀만 실행해도 항상 v5 선형 fade 가 적용되며,
                  패치 셀 실행 순서에 따라 v4(기하 ω)/v5 가 뒤바뀌던
                  이중 평가 문제가 구조적으로 제거됩니다.
        ─────────────────────────────────────────────────────────
        Phase 2 fade (N = n_phase2):
            f(t)      = t / N                        (0 → 1)
            spread(t) = spread0 x (1 - f)            → 종료 시 정확히 0
            b(t)      = b0 + (b_term - b0) x f       (b_term = g_term / Re)
            RI(t)     = spread(t) x BV(t-1)
            BV(t)     = BV(t-1) x (1 + Re x b(t)) + RI(t) x b(t)

        마지막 해 spread=0 → RI_last=0 → compute_valuation 의 TV 가 자동 0.
        (경쟁우위기간 종료 시 초과수익 소멸 = Damodaran/Ohlson 표준 가정)
        """
        re = self._re
        coefs = self.estimate_dupont_coefs()
        eva = self.compute_eva_spread()
        omega, n_phase2, moat_label, n_phase1 = self.moat_to_omega_and_years(eva)

        ph1_df, bv_after_ph1 = self.forecast_roe_phase1(coefs, n_years=n_phase1)
        retention = self.estimate_retention()

        rows = ph1_df.to_dict("records")

        spread0 = float(ph1_df["ri_spread"].iloc[-1]) if not ph1_df.empty else 0.0
        bv_current = bv_after_ph1
        last_year = (int(ph1_df["year"].iloc[-1])
                     if not ph1_df.empty else datetime.now().year)

        # ── fade 목표치 ──────────────────────────────────────
        #  b_term = g_term / Re : 말기 BV 성장률이 g_term(GDP 수준)이 되는 유보율
        g_term = float(np.clip(self.gdp_growth, 0.0, max(re - 0.005, 0.0)))
        b_term = g_term / re if re > 1e-6 else 0.0
        b0 = retention if retention < 0 else min(retention, BV_RETENTION_CAP)

        ri_t = float(ph1_df["ri"].iloc[-1]) if not ph1_df.empty else 0.0

        for t in range(1, n_phase2 + 1):
            f = t / float(n_phase2)                  # 0 → 1
            spread_t = spread0 * (1.0 - f)           # ① 선형 fade → 마지막 해 0
            b_t = b0 + (b_term - b0) * f             # ② retention fade

            if bv_current <= 0:                      # ④ BV 고갈 방어
                bv_current = 0.0
                spread_t = 0.0

            ri_t = spread_t * bv_current
            roe_impl = re + spread_t                 # 단조 수렴 → Re

            rows.append({
                "year":         last_year + t,
                "phase":        "ph2",
                "sales_annual": np.nan, "net_income": np.nan,
                "npm": np.nan, "at": np.nan, "fl": np.nan,
                "roe": roe_impl, "re": re,
                "ri_spread": spread_t,
                "bv_start":  bv_current,
                "ri":        ri_t,
            })

            bv_current = bv_current * (1.0 + re * b_t) + ri_t * b_t

        self.result_df   = pd.DataFrame(rows)
        self._bv_terminal = bv_current
        self._ri_last    = ri_t        # 마지막 해 spread=0 → 0 → TV 자동 0 (③)
        self._omega      = omega       # ★ moat 등급 메타데이터 (감쇠에는 미사용)
        self._moat_label = moat_label
        self._n_phase2   = n_phase2
        self._n_phase1   = len(ph1_df)
        self._eva        = eva
        self._spread0    = spread0
        self._b0         = b0
        self._b_term     = b_term

        if self.verbose:
            log(self.ticker_dg,
                f"RI path(LINEAR-fade v5): Ph1={len(ph1_df)}yr Ph2={n_phase2}yr  "
                f"spread {spread0:.1%}->0  b {b0:.2f}->{b_term:.2f}  "
                f"RI_last={ri_t/1e9:,.1f}B(=0 기대) -> TV=0  [{moat_label}]")
        return self

    def compute_valuation(self):
        re = self._re
        g = self.gdp_growth
        df = self.result_df.copy()

        # BV₀
        wide = self._fs_wide.sort_index()
        eq_ser = wide["total_equity"].dropna()
        bv0_raw = float(eq_ser.iloc[-1]) if not eq_ser.empty else -1.0
        bv0 = bv0_raw if bv0_raw > 0 else self._estimate_ic()

        # PV(RI)
        pv_ri_total = 0.0
        pv_rows = []
        for t, (_, row) in enumerate(df.iterrows(), 1):
            pv = row["ri"] / (1 + re) ** t
            pv_ri_total += pv
            pv_rows.append(pv)
        df["pv_ri"] = pv_rows

        T = len(df)

        # Terminal Value (US v11 방식: RI_last × (1+g) / (Re-g))
        ri_last = self._ri_last
        g_tv = max(re - TV_RE_GAP, self.gdp_growth)
        g_tv = min(g_tv, re - 0.005)
        tv = 0.0
        if ri_last > 0 and re > g_tv:
            tv = ri_last * (1.0 + g_tv) / (re - g_tv)
        elif ri_last > 0:
            tv = ri_last * (1.0 + g_tv) / 0.005
        pv_tv = tv / (1 + re) ** T if tv != 0 else 0.0

        intrinsic = bv0 + pv_ri_total + pv_tv

        # Shares
        shares = np.nan
        for col in ["shares_treasury_adj", "shares_common"]:
            if col in wide.columns:
                s = wide[col].dropna()
                s = s[s > 0]
                if not s.empty:
                    shares = float(s.iloc[-1])
                    break
        if np.isnan(shares):
            self.report.add("shares", "missing", n_obs=0)
            target_price = np.nan
        else:
            target_price = intrinsic / shares

        # Current Price
        cp = load_current_price(self.ticker_dg, self.db_info, table_name=TABLE_PRICE)
        if cp is None:
            self.report.add("current_price", "missing", n_obs=0)
            cp = np.nan

        upside = ((target_price/cp) - 1) * 100 \
                 if (not np.isnan(target_price) and not np.isnan(cp) and cp > 0) else np.nan
        tv_wt = pv_tv / intrinsic * 100 if intrinsic != 0 else np.nan

        # ── ★ v1.1 Sanity 가드 (FCFF v8 ④ 와 동일 취지) ──────────
        #   적정주가/현재가 배율이 허용 범위를 벗어나면 평가 자체를 제외.
        #   (v4 기하 decay 시절 A019180 Upside 95,793% 같은 난센스 차단용
        #    안전망 — v5 선형 fade 로 구조적 발산은 이미 제거됐지만
        #    입력 데이터 이상(주식수/BV/단위)에 대한 2차 방어선으로 유지)
        sanity_ratio = (target_price / cp) if (not np.isnan(target_price)
                                               and not np.isnan(cp) and cp > 0) else np.nan
        if RIM_SANITY_GUARD and not np.isnan(sanity_ratio):
            if (sanity_ratio > RIM_SANITY_MC_RATIO
                    or sanity_ratio < 1.0 / RIM_SANITY_MC_RATIO):
                self.report.add("sanity_guard", "missing", value=sanity_ratio,
                                note=f"TP/CP={sanity_ratio:.1f}배 "
                                     f"(허용 1/{RIM_SANITY_MC_RATIO:.0f}~"
                                     f"{RIM_SANITY_MC_RATIO:.0f}배)")
                raise ValueError(
                    f"[{self.ticker_dg}] v1.1 sanity guard: 적정주가 {target_price:,.0f}원 "
                    f"vs 현재가 {cp:,.0f}원 = {sanity_ratio:.1f}배 "
                    f"(허용 ±{RIM_SANITY_MC_RATIO:.0f}배) -> 평가 제외. "
                    f"입력(주식수/BV/단위) 점검 필요")

        self.result_df = df
        self.valuation = {
            "ticker": self.ticker_dg,
            "re": re, "g_terminal": g_tv,
            "bv0": bv0, "bv_source": self._bv_source,
            "pv_ri": pv_ri_total, "terminal_value": tv, "pv_tv": pv_tv,
            "tv_weight_pct": tv_wt,
            "intrinsic_value": intrinsic,
            "shares": shares, "target_price": target_price,
            "current_price": cp, "upside_pct": upside,
            "moat_label": self._moat_label,
            "omega": self._omega,          # ★ moat 등급 메타 (v5 에선 감쇠에 미사용)
            "fade_mode": "linear_v5",      # ★ Phase 2 감쇠 방식 식별자
            "spread0": getattr(self, "_spread0", np.nan),
            "b0": getattr(self, "_b0", np.nan),
            "b_term": getattr(self, "_b_term", np.nan),
            "sanity_ratio": sanity_ratio,
            "n_phase1": self._n_phase1, "n_phase2": self._n_phase2,
            "eva_spread": self._eva.get("eva_spread", np.nan),
            "roic": self._eva.get("roic", np.nan),
            "roe_re_spread": self._eva.get("roe_re_spread", np.nan),
            "beta_raw": self._beta_info.get("beta_raw") if self._beta_info else np.nan,
            "beta_blume": self._beta_info.get("beta_blume") if self._beta_info else np.nan,
            "revenue_quarters": len(self._sales_actual),
            "forecast_model": self._used_model,
            "forecast_date": self._forecast_date,
        }

        if self.verbose:
            tp_s = f"{target_price:,.0f}원" if not np.isnan(target_price) else "N/A"
            cp_s = f"{cp:,.0f}원" if not np.isnan(cp) else "N/A"
            up_s = f"{upside:+.1f}%" if not np.isnan(upside) else "N/A"
            log(self.ticker_dg,
                f"RIM  BV={bv0/1e12:.2f}조 PV(RI)={pv_ri_total/1e12:.2f}조 "
                f"PV(TV)={pv_tv/1e12:.2f}조 IV={intrinsic/1e12:.2f}조  "
                f"TP={tp_s} CP={cp_s} Up={up_s}  [{self._moat_label}]")
        return self

    # ─────────────────────────────────────────────────────────
    # 8. DB 저장
    # ─────────────────────────────────────────────────────────
    def save_to_db(self, run_date=None):
        if self.result_df is None or self.valuation is None:
            return 0
        run_date = run_date or datetime.now().strftime("%Y-%m-%d")
        v = self.valuation
        n = self._n
        df = self.result_df.copy()

        rows = []
        for _, row in df.iterrows():
            rows.append({
                "date": run_date, "ticker": self.ticker_dg,
                "year_label": str(row.get("year", "")),
                "phase": row.get("phase", ""),
                "sales_forecast": n(row.get("sales_annual")),
                "npm_forecast":   n(row.get("npm")),
                "asset_turnover": n(row.get("at")),
                "fin_leverage":   n(row.get("fl")),
                "roe_forecast":   n(row.get("roe")),
                "re":             n(row.get("re")),
                "ri_spread":      n(row.get("ri_spread")),
                "bv_start":       n(row.get("bv_start")),
                "ri":             n(row.get("ri")),
                "pv_ri":          n(row.get("pv_ri")),
                "moat_label":     v.get("moat_label"),
                "rho":            n(v.get("omega")),
                "n_phase2":       n(v.get("n_phase2")),
                "g_terminal":     n(v.get("g_terminal")),
                "pv_all_ri":      n(v.get("pv_ri")),
                "terminal_value": n(v.get("terminal_value")),
                "intrinsic_value":n(v.get("intrinsic_value")),
                "current_price":  n(v.get("current_price")),
                "upside_pct":     n(v.get("upside_pct")),
                "target_price":   n(v.get("target_price")),
                "beta_raw":       n(v.get("beta_raw")),
                "beta_blume":     n(v.get("beta_blume")),
                "bv_source":      v.get("bv_source"),
                "revenue_quarters": v.get("revenue_quarters"),
                "forecast_model": v.get("forecast_model"),
                "forecast_date":  v.get("forecast_date"),
                "fade_mode":      v.get("fade_mode"),
                "sanity_ratio":   n(v.get("sanity_ratio")),
                "spread0":        n(v.get("spread0")),
            })

        sql = f"""
            INSERT INTO `{TABLE_RESULT}`
            (date, ticker, year_label, phase,
             sales_forecast, npm_forecast, asset_turnover, fin_leverage,
             roe_forecast, re, ri_spread, bv_start, ri, pv_ri,
             moat_label, rho, n_phase2, g_terminal,
             pv_all_ri, terminal_value, intrinsic_value,
             current_price, upside_pct, target_price,
             beta_raw, beta_blume, bv_source,
             revenue_quarters, forecast_model, forecast_date,
             fade_mode, sanity_ratio, spread0)
            VALUES
            (%(date)s, %(ticker)s, %(year_label)s, %(phase)s,
             %(sales_forecast)s, %(npm_forecast)s, %(asset_turnover)s, %(fin_leverage)s,
             %(roe_forecast)s, %(re)s, %(ri_spread)s, %(bv_start)s, %(ri)s, %(pv_ri)s,
             %(moat_label)s, %(rho)s, %(n_phase2)s, %(g_terminal)s,
             %(pv_all_ri)s, %(terminal_value)s, %(intrinsic_value)s,
             %(current_price)s, %(upside_pct)s, %(target_price)s,
             %(beta_raw)s, %(beta_blume)s, %(bv_source)s,
             %(revenue_quarters)s, %(forecast_model)s, %(forecast_date)s,
             %(fade_mode)s, %(sanity_ratio)s, %(spread0)s)
            ON DUPLICATE KEY UPDATE
                ri=VALUES(ri), roe_forecast=VALUES(roe_forecast),
                target_price=VALUES(target_price), upside_pct=VALUES(upside_pct),
                intrinsic_value=VALUES(intrinsic_value),
                moat_label=VALUES(moat_label),
                fade_mode=VALUES(fade_mode), sanity_ratio=VALUES(sanity_ratio)
        """
        conn = get_pymysql_conn(self.db_info)
        try:
            with conn.cursor() as cur:
                cur.executemany(sql, rows)
            conn.commit()
        except Exception:
            conn.rollback(); raise
        finally:
            conn.close()
        return len(rows)

    # ─────────────────────────────────────────────────────────
    # 9. 시각화 (한국 단위 조원)
    # ─────────────────────────────────────────────────────────
    def plot(self):
        if self.result_df is None or self.valuation is None:
            print("[WARN] run() 먼저 실행 필요"); return

        df = self.result_df.copy()
        v = self.valuation
        re = v["re"]
        UNIT, UNIT_LBL = 1e12, "조원"

        fig, axes = plt.subplots(2, 2, figsize=(16, 10))
        fig.suptitle(f"{self.ticker_dg}  Korea RIM Valuation  [{v['moat_label']}]",
                     fontsize=14, fontweight="bold")

        # (0,0) ROE vs Re
        ax = axes[0, 0]
        current_year = datetime.now().year
        wide = self._fs_wide
        roe_h_df = None
        if "net_income" in wide.columns and "total_equity" in wide.columns:
            df_h = wide[["net_income", "total_equity"]].dropna().sort_index()
            df_h = df_h[df_h["total_equity"] > 0].tail(20)
            if len(df_h) >= 4:
                roe_hist = []
                for i in range(3, len(df_h)):
                    ni_ttm = float(df_h["net_income"].iloc[i-3:i+1].sum())
                    eq_avg = float(df_h["total_equity"].iloc[i-3:i+1].mean())
                    if eq_avg <= 0: continue
                    dt = df_h.index[i]
                    yr = dt.year + (dt.month - 1) / 12
                    roe_hist.append({"yr": yr, "roe": ni_ttm / eq_avg})
                roe_h_df = pd.DataFrame(roe_hist)
                ax.plot(roe_h_df["yr"], roe_h_df["roe"] * 100,
                        color="#2980b9", lw=1.5, marker="o", ms=3,
                        label="Historical ROE (TTM)")
                ax.axvline(current_year, color="gray", lw=1, ls="--")

        ph1 = df[df["phase"].str.startswith("ph1")]
        ph2 = df[df["phase"] == "ph2"]
        ph1_x = [current_year + t for t in range(1, len(ph1) + 1)]
        ph2_x = [current_year + len(ph1) + t for t in range(1, len(ph2) + 1)]
        ax.plot(ph1_x, ph1["roe"] * 100, "s-", color="#27ae60", lw=2,
                label="Phase1 ROE (DuPont)")
        ax.plot(ph2_x, ph2["roe"] * 100, "^--", color="#8e44ad", lw=2,
                label="Phase2 ROE (AR(1))")
        ax.axhline(re * 100, color="#e74c3c", lw=1.5, ls=":",
                   label=f"Re={re*100:.1f}%")
        ax.axhline(0, color="black", lw=0.6)
        ax.set_title("ROE vs Cost of Equity (Re)")
        ax.set_ylabel("%")
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f"{x:.0f}%"))
        ax.legend(fontsize=8); ax.grid(alpha=0.3)

        # (0,1) RI time-series
        ax = axes[0, 1]
        ph1_ri = ph1["ri"].values / UNIT
        ph2_ri = ph2["ri"].values / UNIT
        yr_ph1 = list(range(1, len(ph1)+1))
        yr_ph2 = list(range(len(ph1)+1, len(ph1)+len(ph2)+1))
        c_ph1 = ["#27ae60" if r >= 0 else "#e74c3c" for r in ph1_ri]
        c_ph2 = ["#8e44ad" if r >= 0 else "#c0392b" for r in ph2_ri]
        ax.bar(yr_ph1, ph1_ri, color=c_ph1, alpha=0.85, label="Phase1 RI", edgecolor="white")
        ax.bar(yr_ph2, ph2_ri, color=c_ph2, alpha=0.65, label="Phase2 RI", edgecolor="white")
        ax.axhline(0, color="black", lw=0.8)
        ax.axvline(len(ph1) + 0.5, color="gray", lw=1, ls=":")
        pv_tv_u = v["pv_tv"] / UNIT
        ax.axhline(pv_tv_u, color="#e67e22", lw=1.5, ls="--",
                   label=f"PV(TV)={pv_tv_u:.2f}{UNIT_LBL}")
        ax.set_title(f"Residual Income by Year ({UNIT_LBL})")
        ax.set_xlabel("Forecast Year"); ax.set_ylabel(UNIT_LBL)
        ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)

        # (1,0) Value composition
        ax = axes[1, 0]
        bv0_u = v["bv0"] / UNIT
        pv_ri_u = v["pv_ri"] / UNIT
        pv_tv_u = v["pv_tv"] / UNIT
        iv_u = v["intrinsic_value"] / UNIT
        components = {"Book Value": bv0_u, "PV(RI)": pv_ri_u, "PV(TV)": pv_tv_u}
        colors_bar = {"Book Value": "#3498db", "PV(RI)": "#27ae60", "PV(TV)": "#e67e22"}
        bottom = 0.0
        for comp, val in components.items():
            if val > 0:
                ax.bar("Intrinsic Value", val, bottom=bottom,
                       color=colors_bar[comp], alpha=0.85, edgecolor="white",
                       label=f"{comp} {val:.2f}{UNIT_LBL}")
                ax.text(0, bottom + val/2, f"{comp}\n{val:.2f}{UNIT_LBL}",
                        ha="center", va="center", fontsize=9,
                        color="white", fontweight="bold")
                bottom += val
        if (not np.isnan(v["current_price"]) and not np.isnan(v["shares"])
            and v["shares"] > 0):
            mkt_cap_u = v["current_price"] * v["shares"] / UNIT
            ax.axhline(mkt_cap_u, color="#e74c3c", lw=2, ls="--",
                       label=f"Market Cap {mkt_cap_u:.2f}{UNIT_LBL}")
        ax.set_title(f"Intrinsic Value Composition (IV={iv_u:.2f}{UNIT_LBL})")
        ax.set_ylabel(UNIT_LBL); ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)

        # (1,1) Summary text
        ax = axes[1, 1]; ax.axis("off")
        tp = v["target_price"]; cp = v["current_price"]; up = v["upside_pct"]
        evs = v["eva_spread"]; rrs = v["roe_re_spread"]

        def _f(x, u="원", d=0):
            if x is None or (isinstance(x, float) and np.isnan(x)): return "N/A"
            if u == "원":  return f"{x:,.{d}f}원"
            if u == "조":  return f"{x/1e12:,.2f}조원"
            if u == "%":   return f"{x*100:.2f}%"
            return str(x)

        lines = [
            f"Current Price  : {_f(cp)}",
            f"Target Price   : {_f(tp)}",
            f"Upside         : {up:+.1f}%" if not np.isnan(up) else "Upside : N/A",
            "─" * 32,
            f"Book Value     : {_f(v['bv0'],'조')}  [{v['bv_source']}]",
            f"PV(RI)         : {_f(v['pv_ri'],'조')}",
            f"PV(TV)         : {_f(v['pv_tv'],'조')}",
            f"Intrinsic Val  : {_f(v['intrinsic_value'],'조')}",
            "─" * 32,
            f"Re             : {_f(re,'%')}",
            f"g_terminal     : {_f(v['g_terminal'],'%')}",
            f"β_blume        : {v['beta_blume']:.3f}" if not np.isnan(v['beta_blume']) else "β : N/A",
            f"Moat           : {v['moat_label']}",
            f"omega (AR1)    : {v['omega']:.3f}",
            f"Phase1 / Ph2   : {v['n_phase1']} / {v['n_phase2']} yr",
            "─" * 32,
            f"EVA spread     : {evs*100:+.1f}%" if not np.isnan(evs) else "EVA : N/A",
            f"ROE-Re spread  : {rrs*100:+.1f}%" if not np.isnan(rrs) else "ROE-Re : N/A",
        ]
        ax.text(0.05, 0.97, "\n".join(lines), transform=ax.transAxes,
                fontsize=9, verticalalignment="top", fontfamily="monospace",
                bbox=dict(boxstyle="round,pad=0.5", facecolor="#f8f9fa", alpha=0.8))
        plt.tight_layout(); plt.show()

    # ─────────────────────────────────────────────────────────
    # 10. 전체 실행
    # ─────────────────────────────────────────────────────────
    def run(self):
        self.load_sales()
        self.load_financials()
        self.load_re()
        self.compute_ri_path()
        self.compute_valuation()
        return self


# ── Wrapper ────────────────────────────────────────────────────
def process_one_ticker_rim_kr(ticker, engine, db_info, rf, e_rm, kospi_series,
                               verbose=False, run_date=None, save_db=True):
    run_date = run_date or datetime.now().strftime("%Y-%m-%d")
    try:
        m = KoreaRIMModel(ticker=ticker, engine=engine, db_info=db_info,
                          rf=rf, e_rm=e_rm, kospi_series=kospi_series,
                          verbose=verbose)
        m.run()
        rows = m.save_to_db(run_date) if save_db else 0
        v = m.valuation
        return {
            "status": "ok", "ticker": m.ticker_dg,
            "target_price": v.get("target_price", np.nan),
            "current_price": v.get("current_price", np.nan),
            "upside_pct":   v.get("upside_pct", np.nan),
            "re":           v.get("re", np.nan),
            "moat_label":   v.get("moat_label", ""),
            "omega":        v.get("omega", np.nan),
            "n_phase2":     v.get("n_phase2", 0),
            "eva_spread":   v.get("eva_spread", np.nan),
            "bv_source":    v.get("bv_source", ""),
            "rows_saved":   rows,
            "report":       m.report,
        }
    except Exception as e:
        return {
            "status": "fail", "ticker": to_dg_ticker(ticker),
            "target_price": np.nan, "current_price": np.nan, "upside_pct": np.nan,
            "re": np.nan, "moat_label": "", "omega": np.nan, "n_phase2": 0,
            "eva_spread": np.nan, "bv_source": "", "rows_saved": 0,
            "report": None, "msg": str(e)[:150],
        }
    finally:
        gc.collect()


print("[OK] KoreaRIMModel v1.1 정의 완료")
print("     Phase 2 : LINEAR-fade (v5) — 클래스 본문에 내장 (별도 패치 셀 불필요)")
print(f"     Sanity  : TP/CP {RIM_SANITY_MC_RATIO:.0f}배 초과 시 평가 제외 "
      f"(활성={RIM_SANITY_GUARD})")
print("     ※ 이 셀은 클래스 정의만 수행합니다 — 평가는 실행 셀에서만 이루어집니다.")


## Cell 8 · 배치 실행 → DB 저장

- 체크포인트: 완료/실패 종목을 `_korea_rim_checkpoint/` 에 기록 (`SKIP_DONE=True` 로 이어하기)
- 종료 후 데이터 품질 리포트를 `TABLE_QUALITY` 에 `model='RIM'` 으로 저장

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  배치 실행 → DB 저장
#  - ★ 평가는 process_one_ticker_rim_kr() → KoreaRIMModel.run() 한 곳에서만
#  - 체크포인트: 완료/실패 종목을 CHECKPOINT_DIR 에 기록 (SKIP_DONE=True 로 이어하기)
# ═══════════════════════════════════════════════════════════════

if RUN_TICKERS_OVERRIDE:
    RUN_TICKERS = [to_dg_ticker(t) for t in RUN_TICKERS_OVERRIDE]
    log("BATCH", f"OVERRIDE 모드: 지정 종목 {len(RUN_TICKERS)}개만 실행")
else:
    RUN_TICKERS = KOREA_TICKER_LIST[TICKER_START:TICKER_END]

total = len(RUN_TICKERS)
run_date = datetime.now().strftime("%Y-%m-%d")

done_set = set()
if SKIP_DONE and os.path.exists(DONE_PATH):
    with open(DONE_PATH, encoding="utf-8") as f:
        done_set = {l.strip() for l in f if l.strip()}

ok_cnt = skip_cnt = fail_cnt = 0
results, all_reports = [], []
t0 = time.time()

log("BATCH", f"RIM 배치 시작: {total:,}개  run_date={run_date}  (Phase2=LINEAR fade v5)")
print("=" * 84)

for idx, ticker in enumerate(RUN_TICKERS, 1):
    pct = idx / total * 100
    prefix = f"[{idx:>5}/{total}] ({pct:5.1f}%) {ticker:<8}"

    if SKIP_DONE and ticker in done_set:
        print(f"{prefix} SKIP", flush=True)
        skip_cnt += 1
        continue

    res = process_one_ticker_rim_kr(
        ticker, engine, db_info, RF, E_RM, KOSPI_PX,
        verbose=VERBOSE_BATCH, run_date=run_date, save_db=True)
    results.append(res)
    if res["report"] is not None:
        all_reports.append(res["report"])

    if res["status"] == "ok":
        tp, up = res["target_price"], res["upside_pct"]
        tp_s = f"TP={tp:,.0f}원" if not np.isnan(tp) else "TP=N/A"
        up_s = f"{up:+.1f}%" if not np.isnan(up) else ""
        print(f"{prefix} OK  {tp_s} {up_s}  Re={res['re']:.3%}  "
              f"[{res['moat_label']}] Ph2={res['n_phase2']}yr", flush=True)
        with open(DONE_PATH, "a", encoding="utf-8") as f:
            f.write(ticker + "\n")
        ok_cnt += 1
    else:
        print(f"{prefix} FAIL  {res.get('msg', '')}", flush=True)
        with open(FAIL_PATH, "a", encoding="utf-8") as f:
            f.write(f"{ticker}\t{res.get('msg', '')}\n")
        fail_cnt += 1

elapsed = time.time() - t0
print("=" * 84)
log("BATCH", f"완료  OK={ok_cnt}  SKIP={skip_cnt}  FAIL={fail_cnt}  "
             f"경과={elapsed:.0f}s  평균={elapsed / max(ok_cnt + fail_cnt, 1):.1f}s/ticker")

# ── 데이터 품질 로그 저장 ────────────────────────────────────
if all_reports:
    n_q = save_quality_report_to_db(
        all_reports, db_info, table_name=TABLE_QUALITY,
        run_date=run_date, model_name="RIM")
    log("QUALITY", f"품질 로그 {n_q:,}건 → {TABLE_QUALITY}")

# ── 요약 ─────────────────────────────────────────────────────
if results:
    summary = pd.DataFrame([{
        "ticker": r["ticker"], "re": r["re"],
        "moat": r["moat_label"], "n_phase2": r["n_phase2"],
        "tp": r["target_price"], "cp": r["current_price"],
        "upside": r["upside_pct"], "bv_src": r["bv_source"],
        "status": r["status"],
    } for r in results])
    ok_summary = summary[summary["status"] == "ok"].sort_values("upside", ascending=False)
    print("\n[상위 업사이드 TOP 20]")
    display(ok_summary.head(20))

clear_memory()


## Cell 9 · 결과 조회 & 시각화 (업사이드 분포·Moat별 평균)

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  결과 조회 & 시각화
# ═══════════════════════════════════════════════════════════════

# ── 1. DB 현황 (날짜별 평가 기록) ────────────────────────────
conn = get_pymysql_conn(db_info)
try:
    with conn.cursor() as cur:
        cur.execute(f"""
            SELECT DATE(date) AS run_date,
                   COUNT(DISTINCT ticker) AS tickers,
                   COUNT(*) AS total_rows,
                   AVG(target_price) AS avg_tp,
                   AVG(upside_pct) AS avg_upside,
                   AVG(re) AS avg_re
            FROM {TABLE_RESULT}
            GROUP BY DATE(date)
            ORDER BY run_date DESC LIMIT 10
        """)
        db_status_df = pd.DataFrame(cur.fetchall())
finally:
    conn.close()

print("=" * 70); print(f"[DB 현황] {TABLE_RESULT}"); print("=" * 70)
display(db_status_df)

# ── 2. 최신 평가일 결과 ──────────────────────────────────────
conn = get_pymysql_conn(db_info)
try:
    with conn.cursor() as cur:
        cur.execute(f"SELECT MAX(date) AS m FROM {TABLE_RESULT}")
        max_date = cur.fetchone()["m"]
        if max_date:
            cur.execute(f"""
                SELECT ticker,
                       MAX(target_price)    AS target_price,
                       MAX(current_price)   AS current_price,
                       MAX(upside_pct)      AS upside_pct,
                       MAX(re)              AS re,
                       MAX(n_phase2)        AS n_phase2,
                       MAX(moat_label)      AS moat,
                       MAX(bv_source)       AS bv_source,
                       MAX(fade_mode)       AS fade_mode,
                       MAX(sanity_ratio)    AS sanity_ratio,
                       MAX(intrinsic_value) AS intrinsic_value
                FROM {TABLE_RESULT}
                WHERE date = %s
                  AND target_price IS NOT NULL
                  AND current_price IS NOT NULL
                GROUP BY ticker
                ORDER BY upside_pct DESC
            """, (max_date,))
            results_df = pd.DataFrame(cur.fetchall())
        else:
            results_df = pd.DataFrame()
finally:
    conn.close()

if not results_df.empty:
    print(f"\n[최신 평가일: {max_date}] 총 {len(results_df):,}개")
    _modes = results_df["fade_mode"].fillna("(구버전)").value_counts()
    print(f"  Phase2 감쇠 방식 분포: {dict(_modes)}")
    print("\n[업사이드 TOP 20]"); display(results_df.head(20))
    print("\n[다운사이드 TOP 20]"); display(results_df.tail(20))

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"Korea RIM Valuation Summary ({max_date})", fontsize=13)

    ax = axes[0]
    ax.hist(results_df["upside_pct"].dropna().clip(-200, 200), bins=40,
            color="#3498db", edgecolor="white", alpha=0.8)
    ax.axvline(0, color="black", lw=1)
    ax.axvline(20, color="green", lw=1, ls="--", label="+20%")
    ax.set_title("Upside Distribution"); ax.set_xlabel("Upside (%)")
    ax.legend(); ax.grid(alpha=0.3)

    ax = axes[1]
    grade = pd.cut(results_df["upside_pct"],
                   bins=[-np.inf, -20, 0, 20, 50, np.inf],
                   labels=["Strong Sell", "Sell", "Hold", "Buy", "Strong Buy"])
    grade.value_counts().sort_index().plot(
        kind="barh", ax=ax,
        color=["#c0392b", "#e74c3c", "#f39c12", "#2ecc71", "#27ae60"])
    ax.set_title("Valuation Grade"); ax.grid(axis="x", alpha=0.3)

    ax = axes[2]
    moat_stats = (results_df.groupby("moat")["upside_pct"]
                  .agg(["mean", "count"]).sort_values("mean", ascending=False))
    ax.barh(range(len(moat_stats)), moat_stats["mean"], color="#2980b9", alpha=0.8)
    ax.set_yticks(range(len(moat_stats)))
    ax.set_yticklabels([f"{m} (n={int(c)})" for m, c in
                        zip(moat_stats.index, moat_stats["count"])], fontsize=9)
    ax.axvline(0, color="black", lw=1)
    ax.set_title("Avg Upside by Moat"); ax.grid(axis="x", alpha=0.3)

    plt.tight_layout(); plt.show()
else:
    print("[INFO] 데이터 없음 — 배치 실행 셀을 먼저 수행하세요")


## Cell 10 · 데이터 품질 진단

1. 항목별 status 분포 (`ok / fallback_median / fallback_zero / missing`)
2. fallback 비율 TOP 항목
3. 문제 종목 TOP 30 — `inspect_rim_quality("A005930")` 로 종목별 상세 조회 가능

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  데이터 품질 진단 (model='RIM')
# ═══════════════════════════════════════════════════════════════
conn = get_pymysql_conn(db_info)
try:
    with conn.cursor() as cur:
        cur.execute(f"""
            SELECT field, status, COUNT(*) AS cnt,
                   AVG(r_squared) AS avg_r2, AVG(value) AS avg_value
            FROM {TABLE_QUALITY}
            WHERE model='RIM'
              AND date=(SELECT MAX(date) FROM {TABLE_QUALITY} WHERE model='RIM')
            GROUP BY field, status
            ORDER BY field, status
        """)
        df_status = pd.DataFrame(cur.fetchall())

        cur.execute(f"""
            SELECT field,
                   SUM(status='ok') AS ok_cnt,
                   SUM(status='fallback_median') AS fb_med,
                   SUM(status='fallback_zero') AS fb_zero,
                   SUM(status='missing') AS miss,
                   COUNT(*) AS total,
                   ROUND(SUM(status LIKE 'fallback%')/COUNT(*)*100, 1) AS fb_pct
            FROM {TABLE_QUALITY}
            WHERE model='RIM'
              AND date=(SELECT MAX(date) FROM {TABLE_QUALITY} WHERE model='RIM')
            GROUP BY field
            ORDER BY fb_pct DESC
        """)
        df_field = pd.DataFrame(cur.fetchall())

        cur.execute(f"""
            SELECT ticker,
                   SUM(status LIKE 'fallback%') AS fallback_cnt,
                   SUM(status='missing') AS miss_cnt,
                   GROUP_CONCAT(DISTINCT field ORDER BY field SEPARATOR ',') AS issues
            FROM {TABLE_QUALITY}
            WHERE model='RIM'
              AND date=(SELECT MAX(date) FROM {TABLE_QUALITY} WHERE model='RIM')
              AND (status LIKE 'fallback%' OR status='missing')
            GROUP BY ticker
            HAVING fallback_cnt + miss_cnt >= 3
            ORDER BY (fallback_cnt + miss_cnt*2) DESC
            LIMIT 30
        """)
        df_problem = pd.DataFrame(cur.fetchall())
finally:
    conn.close()

print("=" * 70); print("[1] 항목별 status 분포"); print("=" * 70); display(df_status)
print("\n" + "=" * 70); print("[2] 항목별 fallback 비율"); print("=" * 70); display(df_field)
print("\n" + "=" * 70); print("[3] 문제 많은 종목 TOP 30"); print("=" * 70); display(df_problem)


def inspect_rim_quality(ticker: str, run_date: Optional[str] = None) -> pd.DataFrame:
    """특정 종목의 RIM 모델 데이터 품질 상세."""
    tk = to_dg_ticker(ticker)
    conn = get_pymysql_conn(db_info)
    try:
        with conn.cursor() as cur:
            if run_date:
                cur.execute(f"""SELECT date, field, status, n_obs, value, r_squared, note
                                  FROM {TABLE_QUALITY}
                                  WHERE ticker=%s AND date=%s AND model='RIM'
                                  ORDER BY field""", (tk, run_date))
            else:
                cur.execute(f"""SELECT date, field, status, n_obs, value, r_squared, note
                                  FROM {TABLE_QUALITY}
                                  WHERE ticker=%s AND model='RIM'
                                    AND date=(SELECT MAX(date) FROM {TABLE_QUALITY}
                                               WHERE ticker=%s AND model='RIM')
                                  ORDER BY field""", (tk, tk))
            return pd.DataFrame(cur.fetchall())
    finally:
        conn.close()


print("\n[예시] inspect_rim_quality('A005930')")
display(inspect_rim_quality("A005930"))


## Cell 11 · Upside 랭킹 & 지표 테이블

지정 측정일(`RANK_DATES_INPUT`, Cell 2) 기준 종목별 최신 1행으로 집계 후
upside 상위 `TOP_N` 종목 추출 → CSV 저장.

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  측정일자 조회 + 날짜 리스트 기반 Upside 랭킹
#  - 같은 종목이 여러 날 측정됐으면 '가장 최근 측정일' 행만 사용
#    (ROW_NUMBER 윈도우 함수 — MySQL 8.0+ 필요)
#  - RIM 은 종목당 여러 year_label 행이 있으므로 먼저 종목·날짜별로 집약
# ═══════════════════════════════════════════════════════════════

# ── 1. 측정된 날짜 목록 ──────────────────────────────────────
sql_dates = text(f"""
    SELECT `date` AS run_date, COUNT(DISTINCT ticker) AS n_tickers
    FROM `{TABLE_RESULT}`
    GROUP BY `date`
    ORDER BY `date` DESC
""")
dates_df = pd.read_sql(sql_dates, engine)
dates_df["run_date"] = pd.to_datetime(dates_df["run_date"]).dt.strftime("%Y-%m-%d")

print(f"\n{'=' * 60}")
print(f"  측정된 날짜 목록 (총 {len(dates_df)}일)")
print(f"{'=' * 60}")
display(dates_df)

# ── 2. 조회 날짜 결정 (Cell 2 의 RANK_DATES_INPUT) ───────────
RANK_DATES = list(RANK_DATES_INPUT)

if not RANK_DATES:
    RANK_DATES = [dates_df["run_date"].iloc[0]]
    log("RANK", f"날짜 미지정 → 최신 측정일 사용: {RANK_DATES}")
else:
    RANK_DATES = [pd.to_datetime(d).strftime("%Y-%m-%d") for d in RANK_DATES]
    valid = set(dates_df["run_date"])
    missing = [d for d in RANK_DATES if d not in valid]
    if missing:
        log("RANK", f"[WARN] 측정 기록 없는 날짜 (무시됨): {missing}")
    RANK_DATES = [d for d in RANK_DATES if d in valid]
    if not RANK_DATES:
        raise ValueError("입력한 날짜 중 측정 기록이 있는 날짜가 하나도 없습니다.")

log("RANK", f"조회 날짜 {len(RANK_DATES)}일 = {sorted(RANK_DATES, reverse=True)}   TOP_N={TOP_N}")

# ── 3. 종목·날짜별 집약 → 종목별 최신 측정일 1행 ─────────────
_placeholders = ", ".join([f":d{i}" for i in range(len(RANK_DATES))])
_params = {f"d{i}": d for i, d in enumerate(RANK_DATES)}

sql_rank = text(f"""
    SELECT *
    FROM (
        SELECT t.*,
               ROW_NUMBER() OVER (
                   PARTITION BY ticker ORDER BY measured_date DESC
               ) AS rn
        FROM (
            SELECT ticker,
                   `date`               AS measured_date,
                   MAX(target_price)    AS target_price,
                   MAX(current_price)   AS current_price,
                   MAX(upside_pct)      AS upside_pct,
                   MAX(re)              AS re,
                   MAX(g_terminal)      AS g_terminal,
                   MAX(moat_label)      AS moat,
                   MAX(n_phase2)        AS n_phase2,
                   MAX(beta_raw)        AS beta_raw,
                   MAX(beta_blume)      AS beta_blume,
                   MAX(intrinsic_value) AS intrinsic_value,
                   MAX(pv_all_ri)       AS pv_ri,
                   MAX(terminal_value)  AS terminal_value,
                   MAX(bv_source)       AS bv_source,
                   MAX(fade_mode)       AS fade_mode,
                   MAX(sanity_ratio)    AS sanity_ratio,
                   MAX(spread0)         AS spread0
            FROM `{TABLE_RESULT}`
            WHERE `date` IN ({_placeholders})
              AND target_price IS NOT NULL
              AND current_price > 0
              AND upside_pct IS NOT NULL
            GROUP BY ticker, `date`
        ) t
    ) x
    WHERE rn = 1
""")
rank_all = pd.read_sql(sql_rank, engine, params=_params).drop(columns=["rn"])
rank_all["measured_date"] = pd.to_datetime(rank_all["measured_date"]).dt.strftime("%Y-%m-%d")
log("RANK", f"중복 제거 후 유효 종목 {len(rank_all):,}개")

# ── 4. upside 내림차순 상위 N ────────────────────────────────
rank_top = (rank_all.sort_values("upside_pct", ascending=False)
                    .head(TOP_N).reset_index(drop=True))
rank_top.index = rank_top.index + 1
rank_top.index.name = "rank"

disp = pd.DataFrame({
    "ticker":        rank_top["ticker"],
    "측정일":         rank_top["measured_date"],
    "목표주가(원)":    rank_top["target_price"].round(0),
    "현재가격(원)":    rank_top["current_price"].round(0),
    "Upside(%)":     rank_top["upside_pct"].round(1),
    "Moat":          rank_top["moat"],
    "Phase2(yr)":    rank_top["n_phase2"],
    "spread₀(%)":    (rank_top["spread0"] * 100).round(2),
    "Re(%)":         (rank_top["re"] * 100).round(2),
    "g_term(%)":     (rank_top["g_terminal"] * 100).round(2),
    "β_Blume":       rank_top["beta_blume"].round(3),
    "IV(조원)":       (rank_top["intrinsic_value"] / 1e12).round(3),
    "PV(RI)(조원)":   (rank_top["pv_ri"] / 1e12).round(3),
    "TV(조원)":       (rank_top["terminal_value"] / 1e12).round(3),
    "BV출처":         rank_top["bv_source"],
    "TP/CP(배)":      rank_top["sanity_ratio"].round(2),
    "fade":          rank_top["fade_mode"],
})

print(f"\n{'=' * 60}")
print(f"  Upside 랭킹 TOP {min(TOP_N, len(disp))}")
print(f"{'=' * 60}")
display(disp.head(50))

# ── 5. CSV 저장 ──────────────────────────────────────────────
if SAVE_RANK_CSV:
    _tag = "_".join(sorted(RANK_DATES, reverse=True)).replace("-", "")
    _csv = RANK_CSV_DIR / f"korea_rim_rank_{_tag}_top{TOP_N}.csv"
    disp.to_csv(_csv, encoding="utf-8-sig")
    print(f"\n[저장] {_csv}")


## Cell 12 · 종목별 평가 이력 조회

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  종목별 평가 이력 조회 (평가일자별 추이)
# ═══════════════════════════════════════════════════════════════

def get_rim_history(ticker, db_info, table_name=TABLE_RESULT):
    """종목의 날짜별 RIM 평가 요약 (year_label 중 대표 1행으로 집계)."""
    tk = to_dg_ticker(ticker)
    sql = f"""
        SELECT `date`,
               MAX(target_price)    AS target_price,
               MAX(current_price)   AS current_price,
               MAX(upside_pct)      AS upside_pct,
               MAX(re)              AS re,
               MAX(g_terminal)      AS g_terminal,
               MAX(moat_label)      AS moat,
               MAX(n_phase2)        AS n_phase2,
               MAX(intrinsic_value) AS intrinsic_value,
               MAX(pv_all_ri)       AS pv_ri,
               MAX(terminal_value)  AS terminal_value,
               MAX(bv_source)       AS bv_source,
               MAX(fade_mode)       AS fade_mode
        FROM `{table_name}`
        WHERE ticker = %s
        GROUP BY `date`
        ORDER BY `date` DESC
    """
    conn = get_pymysql_conn(db_info)
    try:
        with conn.cursor() as cur:
            cur.execute(sql, (tk,))
            return pd.DataFrame(cur.fetchall())
    finally:
        conn.close()


HIST_TICKER = "A005930"
print(f"[이력] {HIST_TICKER}")
hist = get_rim_history(HIST_TICKER, db_info)
display(hist)

if len(hist) >= 2:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle(f"{HIST_TICKER} — RIM 평가 이력", fontsize=12)
    hs = hist.sort_values("date")

    ax = axes[0]
    ax.plot(hs["date"], hs["current_price"], marker="o", label="Current Price",
            color="#95a5a6")
    ax.plot(hs["date"], hs["target_price"], marker="s", label="Target Price",
            color="#e74c3c")
    ax.set_title("Price Path"); ax.set_ylabel("원")
    ax.legend(); ax.grid(alpha=0.3)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

    ax = axes[1]
    ax.plot(hs["date"], hs["upside_pct"], marker="o", color="#2980b9")
    ax.axhline(0, color="black", lw=1)
    ax.axhline(20, color="green", lw=1, ls="--", alpha=0.6)
    ax.axhline(-20, color="red", lw=1, ls="--", alpha=0.6)
    ax.set_title("Upside Path (%)"); ax.grid(alpha=0.3)
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

    plt.tight_layout(); plt.show()
